# Fase 2 · Pipeline batch, ML distribuido y MLflow

**Análisis de Big Data · Magíster en Data Science UDD · Proyecto Integrador**
**Dataset:** Ali_Display_Ad_Click (Alimama / Taobao) · **Entrega:** lunes 21 de septiembre de 2026
**Equipo:** Juan José Torres · Claudio Ballerini · Cristian Vargas · Christian Vásquez

**Qué produce.** Cada cifra del informe sale de una celda de aquí y queda en
`resultados_fase2.json`. Es la misma regla de la Fase 1: *ninguna afirmación existe sin una celda
que la imprima*.

**Cómo está armado.** El notebook **escribe el paquete `adbd/`** antes de usarlo (sección 0c). No es
un adorno: la pauta evalúa modularidad y reproducibilidad, y así el mismo código que corre aquí es
el que queda versionado en el repositorio y el que ejecuta `scripts/correr_fase2.py` sin notebook.
Una sola fuente de verdad, no dos copias que se desincronizan.

---

### Lo que la Fase 1 dejó decidido y aquí no se vuelve a discutir

| Decisión de la Fase 1 | Cómo entra en la Fase 2 |
|---|---|
| Parquet + ZSTD particionado por `fecha_local` como frontera | Bronze se **reutiliza** si ya existe y cumple el contrato |
| `time_stamp` está en UTC, no en hora local (UTC+8) | La corrección se aplica una sola vez, en Bronze |
| El segmento **sin perfil** (5,76%) se conserva con flag, no se filtra | `sin_perfil` es un *feature* |
| `brand`: el cast fabricó 246.330 nulos inexistentes | **Descartada** como predictor; sobrevive como `brand_conocida` |
| `pvalue_level` (54,24%) y `new_user_class_level` (32,49%): "se deciden en Fase 2" | Sección 2, **con una medición**. En el cruce los porcentajes son otros —54,80% y 30,97%— y esa diferencia es el punto |
| W3 con `RANGE ... 1 PRECEDING` para no meter fuga | Se mantiene literal como *feature* del slot |
| W2 necesitó desempate `(time_stamp, adgroup_id, pid)` para ser determinista | El mismo desempate ordena las ventanas de fatiga |
| Spark para producción (ALS y streaming no existen fuera de él) | Todo el pipeline es Spark; DuckDB queda para validación cruzada |

### Las cuatro decisiones nuevas de esta fase

1. **Split temporal con un día de burn-in.** `06-may` no se entrena: existe para que los históricos
   del `07-may` no sean nulos. Train `07..11-may`, validación `12-may`, test `13-may` — y el test se
   toca **una** vez.
2. **Ningún feature puede ver el futuro, y hay un control ejecutable que lo verifica.** La Fase 1 ya
   encontró una fuga de este tipo (`CURRENT ROW` en W3); aquí el pipeline **se detiene** si reaparece.
3. **Submuestreo de negativos con recalibración.** 1 de cada 10 negativos, todos los positivos, y la
   probabilidad se corrige. La evaluación va siempre sobre el conjunto **completo**.
4. **La línea base es un modelo, no una frase.** Predecir siempre el CTR histórico fija el piso: sin
   ese número, un AUC-PR de 0,09 no se puede leer.

**Ejecutar:** kernel **`Python (adbd-fase2)`** → `Kernel → Restart & Run All`. Entorno: Colab
(Java 11) o Jupyter con Java 11/17. Duración medida sobre el dataset completo: **51,6 min** en una
estación de 22 núcleos (ver sección 11 · el entorno importa y se declara).

## 0. Setup

### 0a. La raíz del proyecto

Jupyter arranca el kernel en el directorio del *notebook* (`notebooks/`), pero **todas** las rutas
del proyecto son relativas a la raíz: `./datos_csv`, `./adbd`, `./mlruns`, `./artefactos_fase2`.
Sin este `chdir` el *notebook* no encuentra los CSV —se va por la rama de descarga y falla con un
`ModuleNotFoundError: kagglehub` que no tiene nada que ver con el problema real— y escribe el
paquete `adbd/` **dentro de** `notebooks/`.

La raíz se busca hacia arriba por una marca que el *notebook* no crea (`requirements.txt`). Si no
aparece ninguna —Colab, Databricks—, no se toca nada: ahí el directorio de trabajo ya es el correcto.

In [ ]:
import os, sys

_MARCAS = ("requirements.txt", "datos_csv")
_d, _raiz = os.path.abspath(os.getcwd()), None
while True:
    if any(os.path.exists(os.path.join(_d, _m)) for _m in _MARCAS):
        _raiz = _d
        break
    _padre = os.path.dirname(_d)
    if _padre == _d:          # se llegó a la raíz del disco sin encontrar marca
        break
    _d = _padre

if _raiz and _raiz != os.getcwd():
    os.chdir(_raiz)
print("directorio de trabajo:", os.getcwd())
print("kernel:", sys.executable)

Y las dependencias. En **Colab y Databricks CE** el entorno es efímero y falta casi todo: se instala
lo que falte. En un **entorno local** esta celda **no instala nada**: si falta un paquete, casi
siempre es porque el notebook se abrió con el kernel de otro proyecto, y llenar ese kernel de
`pyspark` no lo arregla — la celda se detiene y dice qué kernel está corriendo y cuál corresponde.

In [ ]:
import importlib.util, subprocess

_REQ = {"pyspark": "pyspark==3.5.4", "mlflow": "mlflow==2.17.2", "duckdb": "duckdb",
        "polars": "polars", "pandas": "pandas", "pyarrow": "pyarrow",
        "kagglehub": "kagglehub", "psutil": "psutil", "matplotlib": "matplotlib"}
_faltan = [v for k, v in _REQ.items() if importlib.util.find_spec(k) is None]
_efimero = ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
            or "DATABRICKS_RUNTIME_VERSION" in os.environ)

if not _faltan:
    print("todas las dependencias presentes · no se instala nada")
elif _efimero:
    print("entorno efímero · instalando:", " ".join(_faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_faltan])
else:
    raise ModuleNotFoundError(
        f"faltan {', '.join(_faltan)} en el kernel {sys.executable}"
        "\n  Este notebook no instala paquetes en un entorno local."
        "\n  · Elegí el kernel 'Python (adbd-fase2)' en el selector (arriba a la derecha)"
        " y volvé a correr desde el principio."
        "\n  · Si de verdad querés usar este intérprete: pip install -r requirements.txt")

### 0b. Acceso a los datos

Dos vías, en este orden: (a) los CSV ya están en `RUTA_CSV`; (b) descarga del espejo de Kaggle.
**La licencia que este trabajo reconoce es la de Tianchi** (`tianchi.aliyun.com/dataset/56`); el
espejo de Kaggle declara `License(s): unknown` y se usa **solo como medio de acceso**, igual que en
la Fase 1.

In [ ]:
import glob, json, time, shutil
import pandas as pd

RUTA_CSV = os.environ.get("ADBD_RUTA_CSV", "./datos_csv")
ARCHIVOS = ["raw_sample.csv", "ad_feature.csv", "user_profile.csv"]

def _hay(ruta):
    return all(glob.glob(os.path.join(ruta, "**", a), recursive=True) for a in ARCHIVOS)

if not _hay(RUTA_CSV):
    import kagglehub
    RUTA_CSV = kagglehub.dataset_download("pavansanagapati/ad-displayclick-data-on-taobaocom")
    print("descargado en", RUTA_CSV)

os.environ["ADBD_RUTA_CSV"] = RUTA_CSV
for a in ARCHIVOS:
    p = glob.glob(os.path.join(RUTA_CSV, "**", a), recursive=True)[0]
    print(f"{a:<18} {os.path.getsize(p)/1e6:8,.1f} MB   {p}")

# behavior_log.csv NO viene en el espejo. Se verifica, no se supone: es la carta de
# escalamiento que la Fase 1 dejó anotada y que la curva de aprendizaje de la sección 7
# va a decidir si conviene jugar.
print("behavior_log.csv presente:",
      bool(glob.glob(os.path.join(RUTA_CSV, "**", "behavior_log.csv"), recursive=True)))

### 0c. El paquete `adbd`

Las celdas que siguen **escriben el repositorio**. Cada módulo tiene una responsabilidad y se puede
correr aislado; eso es lo que hace posible `scripts/correr_fase2.py`, que reproduce toda la fase sin
abrir un notebook.

| Módulo | Responsabilidad |
|---|---|
| `config` | constantes y supuestos declarados: split, umbrales, tarifa, semilla |
| `utilidades` | cronómetro por etapa, tamaños en disco, exportación a JSON |
| `contrato` | esquema declarado; **detiene** la carga si falta una columna requerida |
| `bronze` | CSV → Parquet+ZSTD por `fecha_local`, corrección UTC→UTC+8 |
| `silver` | joins, perfilamiento de calidad y decisiones de imputación **medidas** |
| `gold` | *features* con control de fuga, split temporal, registro de variables |
| `transformadores` | `CodificadorCentinela` y `CorrectorPrior` (Transformers propios) |
| `features` | ensamblado del `Pipeline` de MLlib y lectura de importancias |
| `modelos` | submuestreo, catálogo de experimentos, `CrossValidator`, ALS |
| `evaluacion` | AUC-PR, *lift* por decil, calibración, métricas de *ranking* |
| `seguimiento` | MLflow: qué se registra y la tabla comparativa |

In [ ]:
!mkdir -p adbd scripts tests notebooks artefactos_fase2

In [ ]:
%%writefile adbd/config.py
"""
Constantes del proyecto. Todo lo que el informe cita como "supuesto declarado" vive aquí,
en un solo lugar, para que no haya dos verdades sobre el mismo número.

Regla heredada de la Fase 1: ninguna afirmación se escribe sin una celda que la imprima.
Su corolario para la Fase 2: ninguna constante se escribe dos veces.
"""
from __future__ import annotations

import os

# ---------------------------------------------------------------------------
# Rutas
# ---------------------------------------------------------------------------
RUTA_CSV = os.environ.get("ADBD_RUTA_CSV", "./datos_csv")
RUTA_PARQUET = os.environ.get("ADBD_RUTA_PARQUET", "./datos_parquet")
RUTA_MLRUNS = os.environ.get("ADBD_RUTA_MLRUNS", "./mlruns")
RUTA_ARTEFACTOS = os.environ.get("ADBD_RUTA_ARTEFACTOS", "./artefactos_fase2")

ESPEJO_KAGGLE = "pavansanagapati/ad-displayclick-data-on-taobaocom"
ARCHIVOS = {
    "raw_sample": "raw_sample.csv",
    "ad_feature": "ad_feature.csv",
    "user_profile": "user_profile.csv",
}

# Capas. La frontera es Parquet, como quedó decidido en la Fase 1 (sección 4.2).
BRONZE_IMPRESIONES = f"{RUTA_PARQUET}/impresiones"
BRONZE_ADS = f"{RUTA_PARQUET}/ad_feature"
BRONZE_USR = f"{RUTA_PARQUET}/user_profile"
SILVER = f"{RUTA_PARQUET}/silver_impresiones"
GOLD = f"{RUTA_PARQUET}/gold_ads_v1"

# ---------------------------------------------------------------------------
# Split temporal. NO es aleatorio y la razón es la única que importa:
# en producción el modelo predice el futuro, así que la evaluación tiene que
# hacer lo mismo. Un split aleatorio sobre datos con historia de usuario mezcla
# impresiones del mismo usuario entre train y test y regala señal.
#
# El dataset son 8 días LOCALES completos (2017-05-06 .. 2017-05-13, UTC+8).
#   BURN-IN  06-may : NO se entrena con él. Existe solo para que el primer día
#                     entrenable tenga historia y los features diferidos no sean nulos.
#   TRAIN    07..11-may (5 días)
#   VAL      12-may       selección de modelo e hiperparámetros
#   TEST     13-may       se toca UNA vez, al final
# ---------------------------------------------------------------------------
DIA_BURNIN = "2017-05-06"
TRAIN_INI, TRAIN_FIN = "2017-05-07", "2017-05-11"   # ambos inclusive
DIA_VAL = "2017-05-12"
DIA_TEST = "2017-05-13"

# ---------------------------------------------------------------------------
# Calidad de datos: los umbrales de la Fase 1, sin cambiarlos a conveniencia.
#   <= 1%  -> se puede descartar la fila
#   <= 5%  -> se imputa
#   >  5%  -> NO se filtra ni se imputa a ciegas: categoría explícita + flag,
#             y se MIDE si la ausencia es informativa antes de decidir.
# ---------------------------------------------------------------------------
UMBRAL_DESCARTE = 0.01
UMBRAL_IMPUTACION = 0.05

# ---------------------------------------------------------------------------
# Codificación de objetivo diferida (target encoding con retardo de un día).
# m es el peso del prior en el suavizado de m-estimación:
#     ctr_hist = (clicks_previos + m * p0) / (impresiones_previas + m)
# Con m = 200 una entidad necesita ~200 impresiones históricas para que su propia
# tasa pese la mitad. Es un supuesto declarado y la sección de sensibilidad lo mueve.
# ---------------------------------------------------------------------------
M_SUAVIZADO = 200.0
M_SUAVIZADO_USUARIO = 20.0     # el usuario tiene ~25 impresiones en total: m alto lo anularía

# adgroup_id queda FUERA de esta lista a propósito: 846.811 avisos x 8 días son 6,8M
# de filas de historia que hay que unir contra 26,6M de impresiones, y el nivel
# campaign_id lo contiene jerárquicamente (un adgroup pertenece a una campaña).
# Es una decisión de costo, declarada, no un olvido.
ENTIDADES_HISTORICAS = ["cate_id", "campaign_id", "customer", "pid"]

# ---------------------------------------------------------------------------
# Desbalance. CTR global 5,14% (~1:18). Se submuestrean NEGATIVOS y se
# RECALIBRA la probabilidad; los positivos nunca se tocan.
# ---------------------------------------------------------------------------
TASA_NEGATIVOS = 0.10          # se conserva 1 de cada 10 negativos en entrenamiento
FRACCION_BUSQUEDA = 0.20       # submuestra adicional SOLO para la búsqueda de hiperparámetros
SEMILLA = 42

# ---------------------------------------------------------------------------
# ALS (feedback implícito): usuario x categoría, clicks como confianza.
# ---------------------------------------------------------------------------
ALS_RANK = 32
ALS_REG = 0.05
ALS_ALPHA = 20.0
ALS_ITER = 10
TOP_K = 10

# ---------------------------------------------------------------------------
# FinOps. Misma tarifa que la Fase 1 para que las dos fases sean comparables.
# Es una REFERENCIA de dimensionamiento, no un gasto incurrido: el proyecto
# corre en Colab gratuito y el costo monetario efectivo es US$ 0.
# ---------------------------------------------------------------------------
TARIFA_USD_HORA = 0.27         # e2-standard-8 (8 vCPU / 32 GB), on-demand, ago-2026
FACTOR_BEHAVIOR = 704 / 26.6   # si entra behavior_log, el volumen se multiplica por esto

# Configuración de Spark. shuffle.partitions = 8 es la decisión de la Fase 1 para
# 1,1 GB en 2 núcleos; el default de 200 produce particiones de pocos MB.
SPARK_CONF = {
    "spark.sql.shuffle.partitions": "8",
    # 8g es lo que da Colab. ADBD_DRIVER_MEM permite bajarlo para la prueba de humo
    # o subirlo en Databricks sin tocar el código.
    "spark.driver.memory": os.environ.get("ADBD_DRIVER_MEM", "8g"),
    "spark.sql.session.timeZone": "UTC",
    "spark.sql.adaptive.enabled": "true",
    # Arrow acelera toPandas(). Se puede apagar con ADBD_ARROW=false: la combinación
    # Spark 3.5 + JDK 21 no la soporta (el Arrow que empaqueta Spark 3.5 es anterior),
    # y la prueba de humo de este repositorio corre en esa JVM.
    "spark.sql.execution.arrow.pyspark.enabled": os.environ.get("ADBD_ARROW", "true"),
}

# Arrow toca memoria fuera del heap por reflexión. Desde Java 17 el módulo
# java.nio está cerrado y `toPandas()` muere con
# "sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long,int) not available".
# Colab trae Java 11 y no lo necesita; la JVM tiene que recibir estas banderas
# ANTES de arrancar, así que van por PYSPARK_SUBMIT_ARGS y no por .config()
# (ponerlas ahí en modo local se ignora en silencio, que es peor que fallar).
OPCIONES_JVM = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "-Dio.netty.tryReflectionSetAccessible=true"
)

NOMBRE_EXPERIMENTO = "adbd_fase2_ctr"

In [ ]:
%%writefile adbd/utilidades.py
"""Cronómetro, tamaños en disco y exportación de resultados."""
from __future__ import annotations

import json
import os
import time
from contextlib import contextmanager

from . import config


class Crono:
    """Registro de tiempos con nombre. Reemplaza los %%time sueltos por algo que
    después se pueda tabular, que es lo que faltó en la v1 de la Fase 1."""

    def __init__(self) -> None:
        self.marcas: dict[str, float] = {}

    @contextmanager
    def medir(self, etiqueta: str):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self.marcas[etiqueta] = round(time.perf_counter() - t0, 2)
            print(f"  ⏱  {etiqueta}: {self.marcas[etiqueta]} s")

    def total(self) -> float:
        return round(sum(self.marcas.values()), 2)

    def tabla(self):
        import pandas as pd

        return (
            pd.DataFrame([{"etapa": k, "segundos": v} for k, v in self.marcas.items()])
            .assign(
                minutos=lambda d: (d.segundos / 60).round(2),
                usd_referencial=lambda d: d.segundos.map(usd),
            )
            .sort_values("segundos", ascending=False)
            .reset_index(drop=True)
        )


def usd(segundos: float) -> float:
    """Valoriza tiempo de cómputo contra una VM de mercado. Referencia, no gasto."""
    return round(segundos / 3600 * config.TARIFA_USD_HORA, 4)


def tam_mb(ruta: str) -> float:
    if not os.path.exists(ruta):
        return 0.0
    if os.path.isfile(ruta):
        return os.path.getsize(ruta) / 1e6
    return sum(
        os.path.getsize(os.path.join(d, f))
        for d, _, fs in os.walk(ruta)
        for f in fs
    ) / 1e6


def limpiar(o):
    """numpy/pandas -> tipos JSON. Sin esto json.dump revienta con np.int64.

    NaN e infinitos se convierten en null: `json.dump` los escribe como `NaN`,
    que Python vuelve a leer pero **no es JSON válido** — cualquier otro lector
    (el generador del informe, por ejemplo) falla al parsear el archivo.
    """
    import math

    import numpy as np

    if isinstance(o, dict):
        return {str(k): limpiar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [limpiar(v) for v in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        v = float(o)
        return None if (math.isnan(v) or math.isinf(v)) else v
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def exportar(res: dict, ruta: str = "resultados_fase2.json") -> str:
    res = dict(res)
    res["generado"] = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(limpiar(res), f, ensure_ascii=False, indent=2, default=str,
                  allow_nan=False)
    print(f"{ruta} escrito · {len(res)} claves")
    return ruta


def preparar_hadoop_windows(verboso: bool = True) -> str | None:
    """En Windows, Spark resuelve permisos del sistema de archivos local con
    `winutils.exe` + `hadoop.dll`, que el paquete `pyspark` NO empaqueta. Sin ellos
    la JVM muere antes de crear el directorio temporal del driver:

        java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset

    No es un error del pipeline —el mismo código corre tal cual en Colab y en
    Linux— sino un requisito del entorno, y por eso se resuelve aquí una sola vez
    en lugar de repetirlo en el notebook, en los tests y en el script de CLI.

    Fuera de Windows no hace nada. Devuelve el HADOOP_HOME efectivo, o None.
    """
    import platform

    if platform.system() != "Windows":
        return None

    raiz = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    candidatos = [
        os.environ.get("HADOOP_HOME"),          # lo que el entorno ya declare manda
        os.path.join(raiz, "vendor", "hadoop"),  # copia versionada con el repositorio
        os.path.join(os.environ.get("LOCALAPPDATA", ""), "hadoop"),
        os.path.join(os.path.expanduser("~"), "hadoop"),
        r"C:\hadoop",
    ]
    for c in candidatos:
        if not c or not os.path.isfile(os.path.join(c, "bin", "winutils.exe")):
            continue
        binario = os.path.join(c, "bin")
        os.environ["HADOOP_HOME"] = c
        # hadoop.dll se carga por java.library.path, que en Windows incluye PATH.
        rutas = os.environ.get("PATH", "").split(os.pathsep)
        if not any(os.path.normcase(r) == os.path.normcase(binario) for r in rutas):
            os.environ["PATH"] = binario + os.pathsep + os.environ.get("PATH", "")
        if verboso:
            print(f"  Windows · HADOOP_HOME = {c}")
        return c

    raise RuntimeError(
        "Windows sin los binarios nativos de Hadoop: Spark no puede arrancar.\n"
        "  Faltan winutils.exe y hadoop.dll (Hadoop 3.3.x, el que empaqueta "
        "pyspark 3.5.4).\n"
        f"  Déjalos en {os.path.join(raiz, 'vendor', 'hadoop', 'bin')} "
        "o exporta HADOOP_HOME apuntando a la carpeta que los contiene.\n"
        "  En Linux, macOS y Colab no hace falta nada de esto (ver README)."
    )


def preparar_java(verboso: bool = True) -> str | None:
    """Deja una JVM alcanzable para el lanzador de Spark, que la busca en
    `JAVA_HOME/bin/java` y, si no, en `java` a secas por el PATH.

    El caso que esto resuelve: el JDK está instalado DENTRO del entorno conda
    (`Library/lib/jvm` en Windows, `lib/jvm` en Linux/macOS) y solo entra al PATH
    al hacer `conda activate`. Un kernel de Jupyter lanzado por VSCode o por
    `jupyter lab` desde otro entorno arranca sin esa activación, no encuentra
    `java`, y Spark muere con un mensaje que no dice nada de Java:

        PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited
        before sending its port number.

    Orden: JAVA_HOME si ya está definido y es válido; `java` en el PATH; el JDK
    del entorno del intérprete actual. Si nada aparece, se detiene con el mensaje
    de qué instalar. Devuelve el JAVA_HOME efectivo, o None si se usa el del PATH.
    """
    import shutil
    import sys

    exe = "java.exe" if os.name == "nt" else "java"

    jh = os.environ.get("JAVA_HOME")
    if jh and os.path.isfile(os.path.join(jh, "bin", exe)):
        return jh
    if shutil.which("java"):
        return None

    candidatos = [
        os.path.join(sys.prefix, "Library", "lib", "jvm"),   # conda-forge openjdk, Windows
        os.path.join(sys.prefix, "lib", "jvm"),              # conda-forge openjdk, Linux/macOS
        os.path.join(sys.prefix, "Library"),                 # java.exe directo en Library/bin
        sys.prefix,
    ]
    for c in candidatos:
        if os.path.isfile(os.path.join(c, "bin", exe)):
            os.environ["JAVA_HOME"] = c
            os.environ["PATH"] = os.path.join(c, "bin") + os.pathsep + os.environ.get("PATH", "")
            if verboso:
                print(f"  JAVA_HOME = {c}")
            return c

    raise RuntimeError(
        "No hay una JVM alcanzable: ni JAVA_HOME, ni `java` en el PATH, ni un JDK en "
        f"el entorno {sys.prefix}.\n"
        "  Spark 3.5 necesita Java 11 o 17. En conda: conda install -c conda-forge openjdk=17\n"
        "  Con un JDK instalado aparte: exportar JAVA_HOME apuntando a su carpeta raíz."
    )


def crear_sesion(nombre: str = "ADBD-Fase2"):
    """Sesión de Spark con la configuración declarada en config.SPARK_CONF."""
    from pyspark.sql import SparkSession

    preparar_java()
    preparar_hadoop_windows()

    # Las banderas de módulo tienen que llegar a la JVM antes de que arranque.
    args = os.environ.get("PYSPARK_SUBMIT_ARGS", "")
    if "--add-opens" not in args:
        base = args.replace("pyspark-shell", "").strip()
        os.environ["PYSPARK_SUBMIT_ARGS"] = (
            f'{base} --driver-java-options "{config.OPCIONES_JVM}" pyspark-shell'
        ).strip()

    b = SparkSession.builder.appName(nombre)
    for k, v in config.SPARK_CONF.items():
        b = b.config(k, v)
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def huella_entorno(spark) -> dict:
    """Qué máquina produjo estos números. Sin esto ningún tiempo es interpretable."""
    import multiprocessing
    import platform

    try:
        import psutil

        ram = f"{round(psutil.virtual_memory().total / 1e9, 1)} GB"
    except Exception:
        ram = "no determinada"
    return {
        "so": platform.system(),
        "nucleos": multiprocessing.cpu_count(),
        "ram": ram,
        "spark": spark.version,
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "tz_sesion": spark.conf.get("spark.sql.session.timeZone"),
    }

In [ ]:
%%writefile adbd/contrato.py
"""
Contrato de lectura. Es el mismo de la Fase 1, sin cambios de fondo: el esquema
se declara, no se infiere, y la ausencia de una columna requerida DETIENE la
ejecución en vez de borrarla en silencio.

Dos cosas que este contrato ya se cobró en la Fase 1 y que siguen vigentes:
  1. El espejo publica 'new_user_class_level ' con un espacio final.
  2. El cast a double fabricó 246.330 nulos en ad_feature.brand, que NO existen
     en el origen. Por eso `brand` no entra como predictor en la Fase 2 (ver gold.py):
     modelar esa columna sería modelar un artefacto de la línea de carga.
"""
from __future__ import annotations

import glob
import os
import re

from pyspark.sql import functions as F

from . import config

ESQUEMA = {
    "raw_sample": {
        "user": "bigint", "time_stamp": "bigint", "adgroup_id": "bigint",
        "pid": "string", "nonclk": "int", "clk": "int",
    },
    "ad_feature": {
        "adgroup_id": "bigint", "cate_id": "bigint", "campaign_id": "bigint",
        "customer": "bigint", "brand": "double", "price": "double",
    },
    "user_profile": {
        "userid": "bigint", "cms_segid": "int", "cms_group_id": "int",
        "final_gender_code": "int", "age_level": "int", "pvalue_level": "double",
        "shopping_level": "int", "occupation": "int", "new_user_class_level": "double",
    },
}

REQUERIDAS = {
    "raw_sample": {"user", "time_stamp", "adgroup_id", "pid", "clk"},
    "ad_feature": {"adgroup_id", "cate_id", "campaign_id", "customer", "price"},
    "user_profile": {"userid", "cms_segid", "cms_group_id", "final_gender_code",
                     "age_level", "pvalue_level", "shopping_level", "occupation",
                     "new_user_class_level"},
}


def _clave(c: str) -> str:
    """Normaliza encabezados para comparar: sin BOM, sin espacios, en minúsculas."""
    return re.sub(r"\s+", "", c.replace("﻿", "")).lower()


def localizar(nombre: str, ruta_csv: str | None = None) -> str | None:
    ruta_csv = ruta_csv or config.RUTA_CSV
    hits = glob.glob(os.path.join(ruta_csv, "**", config.ARCHIVOS[nombre]), recursive=True)
    return hits[0] if hits else None


def resolver_csv(ruta_csv: str | None = None) -> dict[str, str]:
    """Devuelve {tabla: ruta}. Si falta alguno, baja el espejo de Kaggle.
    La licencia reconocida es la de Tianchi; el espejo es solo un medio de acceso."""
    ruta_csv = ruta_csv or config.RUTA_CSV
    if not all(localizar(n, ruta_csv) for n in config.ARCHIVOS):
        # El fallo que más confunde aquí no es "falta el dataset" sino
        # "ModuleNotFoundError: kagglehub" a 20 líneas de profundidad, cuando en
        # realidad los CSV estaban en disco y quien llamó apuntó a la carpeta
        # equivocada. El mensaje tiene que decir las dos cosas.
        try:
            import kagglehub
        except ModuleNotFoundError as e:
            raise ModuleNotFoundError(
                f"no hay CSV en {os.path.abspath(ruta_csv)} y kagglehub no esta instalado.\n"
                f"  · directorio de trabajo actual: {os.getcwd()}\n"
                "  · si los datos ya estan en otra carpeta: ADBD_RUTA_CSV=<ruta> "
                "(o config.RUTA_CSV = <ruta>)\n"
                "  · para bajarlos del espejo de Kaggle: pip install kagglehub"
            ) from e

        ruta_csv = kagglehub.dataset_download(config.ESPEJO_KAGGLE)
        print("descargado en", ruta_csv)
    csv = {n: localizar(n, ruta_csv) for n in config.ARCHIVOS}
    faltan = [n for n, p in csv.items() if p is None]
    if faltan:
        raise FileNotFoundError(f"no se encontraron los CSV: {faltan} bajo {ruta_csv}")
    return csv


def leer_csv(spark, nombre: str, csv: dict[str, str], verboso: bool = True):
    """Lee un CSV aplicando el contrato. Devuelve (crudo, casteado, mapa_de_nombres)."""
    cols = ESQUEMA[nombre]
    crudo = spark.read.option("header", True).option("inferSchema", False).csv(csv[nombre])
    reales = {_clave(c): c for c in crudo.columns}
    resuelto, ausentes, saneadas = {}, [], []
    for c, t in cols.items():
        orig = reales.get(_clave(c))
        if orig is None:
            ausentes.append(c)
            continue
        resuelto[c] = (orig, t)
        if orig != c:
            saneadas.append(f"{orig!r} -> {c}")
    faltan = sorted(set(ausentes) & REQUERIDAS[nombre])
    if faltan:
        raise ValueError(
            f"{nombre}: faltan columnas REQUERIDAS: {faltan}. Encabezado real: {crudo.columns}"
        )
    if verboso and ausentes:
        print(f"  aviso · {nombre}: declaradas y ausentes -> {sorted(set(ausentes) - set(faltan))}")
    if verboso and saneadas:
        print(f"  encabezado saneado · {nombre}: {saneadas}")
    df = crudo.select([F.col(f"`{o}`").cast(t).alias(c) for c, (o, t) in resuelto.items()])
    return crudo, df, {c: o for c, (o, _) in resuelto.items()}


def auditar_cast(spark, csv: dict[str, str]):
    """Nulos ANTES vs DESPUÉS del cast. La diferencia la creó la línea de carga,
    no el origen, y hay que saberlo antes de que alguien use esa columna como feature."""
    import pandas as pd

    filas = []
    for nombre in ESQUEMA:
        crudo, cast, mapa = leer_csv(spark, nombre, csv, verboso=False)
        n_antes = crudo.select(
            [F.sum(F.col(f"`{o}`").isNull().cast("long")).alias(c) for c, o in mapa.items()]
        ).collect()[0].asDict()
        n_desp = cast.select(
            [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in mapa]
        ).collect()[0].asDict()
        for c in mapa:
            filas.append((nombre, c, n_antes[c], n_desp[c], n_desp[c] - n_antes[c]))
    return pd.DataFrame(
        filas, columns=["tabla", "columna", "nulos_antes", "nulos_despues", "creados_por_el_cast"]
    )

In [ ]:
%%writefile adbd/bronze.py
"""
BRONZE · CSV crudo -> Parquet + ZSTD particionado por fecha_local.

Es exactamente la capa que produjo la Fase 1 y no se reescribe: si el directorio
ya existe con el contrato cumplido, se reutiliza. La reproducibilidad de extremo
a extremo que pide la Fase 3 exige que esta función pueda reconstruirlo todo
desde los CSV, así que existe y se puede forzar con `rehacer=True`.

La corrección de zona horaria (UTC -> UTC+8) se aplica AQUÍ, una sola vez.
Sin ella todo feature horario del modelo queda desplazado 8 horas.
"""
from __future__ import annotations

import os

from pyspark.sql import functions as F

from . import config, contrato
from .utilidades import tam_mb


def construir(spark, csv: dict[str, str] | None = None, rehacer: bool = False) -> dict:
    """Escribe (o reutiliza) la capa Bronze. Devuelve métricas para el informe."""
    res: dict = {}
    listo = all(
        os.path.exists(p)
        for p in (config.BRONZE_IMPRESIONES, config.BRONZE_ADS, config.BRONZE_USR)
    )
    if listo and not rehacer and _contrato_ok(spark):
        print("Bronze ya existe y cumple el contrato · se reutiliza (Fase 1)")
        res["bronze_reutilizado"] = True
    else:
        csv = csv or contrato.resolver_csv()
        res["bronze_reutilizado"] = False
        import time

        t0 = time.perf_counter()
        raw = (
            contrato.leer_csv(spark, "raw_sample", csv)[1]
            .withColumnRenamed("user", "userid")
            .withColumn("ts_utc", F.to_timestamp(F.from_unixtime("time_stamp")))
            .withColumn("ts_local", F.col("ts_utc") + F.expr("INTERVAL 8 HOURS"))
            .withColumn("fecha_local", F.to_date("ts_local"))
            .withColumn("hora_local", F.hour("ts_local"))
            .withColumn(
                "franja",
                F.when(F.col("hora_local") < 6, "madrugada")
                .when(F.col("hora_local") < 12, "manana")
                .when(F.col("hora_local") < 19, "tarde")
                .otherwise("noche"),
            )
        )
        (raw.write.mode("overwrite").option("compression", "zstd")
            .partitionBy("fecha_local").parquet(config.BRONZE_IMPRESIONES))
        for n, destino in (("ad_feature", config.BRONZE_ADS), ("user_profile", config.BRONZE_USR)):
            (contrato.leer_csv(spark, n, csv)[1].write.mode("overwrite")
                .option("compression", "zstd").parquet(destino))
        res["t_conversion"] = round(time.perf_counter() - t0, 1)
        assert _contrato_ok(spark), "el Parquet escrito no cumple el contrato"

    mb = tam_mb(config.BRONZE_IMPRESIONES)
    particiones = [
        p for p in os.listdir(config.BRONZE_IMPRESIONES) if p.startswith("fecha_local=")
    ]
    res["mb_bronze"] = round(mb, 1)
    res["n_particiones"] = len(particiones)
    res["particiones"] = sorted(p.split("=")[1] for p in particiones)
    print(f"Bronze: {res['mb_bronze']:,} MB en {res['n_particiones']} particiones")
    return res


def _contrato_ok(spark) -> bool:
    """El Parquet de una corrida anterior con esquema incompleto sobrevive en disco.
    Se verifica en vez de suponerlo."""
    try:
        esperado = {
            config.BRONZE_IMPRESIONES: {"userid", "time_stamp", "adgroup_id", "pid", "clk",
                                        "ts_local", "fecha_local", "hora_local", "franja"},
            config.BRONZE_ADS: contrato.REQUERIDAS["ad_feature"],
            config.BRONZE_USR: contrato.REQUERIDAS["user_profile"],
        }
        for ruta, req in esperado.items():
            if not req.issubset(set(spark.read.parquet(ruta).columns)):
                return False
        return True
    except Exception:
        return False


def leer(spark):
    """Las tres tablas Bronze, con vistas SQL registradas."""
    imp = spark.read.parquet(config.BRONZE_IMPRESIONES)
    ads = spark.read.parquet(config.BRONZE_ADS)
    usr = spark.read.parquet(config.BRONZE_USR)
    for df, n in ((imp, "impresiones"), (ads, "ad_feature"), (usr, "user_profile")):
        df.createOrReplaceTempView(n)
    return imp, ads, usr

In [ ]:
%%writefile adbd/silver.py
"""
SILVER · impresiones ⋈ ad_feature ⋈ user_profile, con perfilamiento de calidad
y las decisiones de imputación TOMADAS CON UNA MEDICIÓN, no por costumbre.

La Fase 1 dejó dos columnas explícitamente pendientes para acá:
    pvalue_level          54,24% de nulos
    new_user_class_level  32,49% de nulos
La respuesta refleja no es imputar con la moda. Antes de decidir se mide si la
AUSENCIA es informativa: si el grupo sin dato tiene un CTR distinto del grupo con
dato, entonces "no sé" es información sobre el usuario y sustituirla por la moda
la destruye. `perfilar_calidad()` produce esa evidencia y `decidir_imputacion()`
aplica la regla de umbrales de la Fase 1 sobre ella.

Medido sobre el cruce real, los porcentajes NO son los de la Fase 1 y la
diferencia es del tipo que importa: `pvalue_level` tiene **54,80%** de ausencia en
el cruce (54,24% dentro de la tabla de perfiles) y `new_user_class_level`
**30,97%** (32,49% dentro de la tabla). La completitud de la tabla y la del cruce
son dos números distintos, y el que decide es el segundo, porque es el dato con el
que se entrena.
"""
from __future__ import annotations

import pandas as pd
from pyspark.sql import functions as F

from . import config
from .bronze import leer as leer_bronze

# Columnas de perfil que pueden faltar. -1 es el centinela de "desconocido":
# no es un valor válido de ninguna de ellas, así que no se confunde con un dato real.
CATEGORICAS_PERFIL = [
    "cms_segid", "cms_group_id", "final_gender_code", "age_level",
    "pvalue_level", "shopping_level", "occupation", "new_user_class_level",
]
CENTINELA = -1


def perfilar_calidad(spark) -> pd.DataFrame:
    """% de nulos por columna EN EL CRUCE (no dentro de la tabla de perfiles) y
    CTR del grupo ausente vs el presente. Es la tabla que decide la imputación.

    El hallazgo 1(b) de la Fase 1 fue exactamente esto: dentro de user_profile.csv
    final_gender_code tiene 0% de nulos, pero en el JOIN falta el 5,76%. La
    completitud relevante es la del cruce, porque es el dato con el que se entrena.
    """
    leer_bronze(spark)
    filas = []
    for c in CATEGORICAS_PERFIL:
        r = spark.sql(f"""
            SELECT
              COUNT(*)                                                   AS n,
              SUM(CASE WHEN u.{c} IS NULL THEN 1 ELSE 0 END)             AS n_nulos,
              SUM(CASE WHEN u.{c} IS NULL THEN i.clk ELSE 0 END)         AS clk_nulos,
              SUM(CASE WHEN u.{c} IS NOT NULL THEN i.clk ELSE 0 END)     AS clk_no_nulos
            FROM impresiones i LEFT JOIN user_profile u ON i.userid = u.userid
        """).collect()[0]
        n, nn = int(r["n"]), int(r["n_nulos"])
        ctr_aus = (r["clk_nulos"] / nn * 100) if nn else None
        ctr_pre = (r["clk_no_nulos"] / (n - nn) * 100) if n - nn else None
        filas.append({
            "columna": c,
            "pct_nulos_en_cruce": round(100 * nn / n, 2),
            "impresiones_sin_dato": nn,
            "ctr_sin_dato_pct": round(ctr_aus, 3) if ctr_aus is not None else None,
            "ctr_con_dato_pct": round(ctr_pre, 3) if ctr_pre is not None else None,
            "razon_ctr": round(ctr_aus / ctr_pre, 3) if ctr_aus and ctr_pre else None,
            "z": _z_dos_proporciones(int(r["clk_nulos"]), nn,
                                     int(r["clk_no_nulos"]), n - nn),
        })
    return pd.DataFrame(filas)


def _z_dos_proporciones(exitos_a: int, n_a: int, exitos_b: int, n_b: int):
    """z de la diferencia de dos proporciones, con varianza agrupada.

    Existe porque la primera versión de `decidir_imputacion` usaba un umbral fijo
    —"informativa si el CTR difiere más de 5%"— y sobre 26,6M de filas ese umbral
    está mal calibrado en las dos direcciones: declara *no informativa* una
    diferencia de 5,335% contra 5,132% que con 1,5M de observaciones tiene z = 11,
    y declararía informativa cualquier ruido medido sobre un grupo chico. El
    tamaño del efecto y la evidencia de que el efecto existe son dos preguntas
    distintas, y la regla de imputación necesita la segunda.
    """
    import math

    if not n_a or not n_b:
        return None
    pa, pb = exitos_a / n_a, exitos_b / n_b
    p = (exitos_a + exitos_b) / (n_a + n_b)
    se = math.sqrt(p * (1 - p) * (1 / n_a + 1 / n_b))
    return round((pa - pb) / se, 2) if se else None


def decidir_imputacion(perfil: pd.DataFrame, z_critico: float = 3.0) -> pd.DataFrame:
    """Aplica la regla de umbrales de la Fase 1 sobre la evidencia medida.

    <= 1%  descartar_fila          la ausencia es marginal
    <= 5%  imputar_moda            cuesta poco y no mueve la distribución
    >  5%  categoria_desconocido   + flag

    y en los tres casos, si la ausencia es INFORMATIVA —el CTR del grupo sin dato
    difiere del CTR del grupo con dato más allá del azar— gana
    `categoria_desconocido`, porque imputar destruiría esa señal.

    **Qué decide "informativa", y por qué no es el tamaño del efecto.** La primera
    versión de esta función usaba un umbral fijo sobre la razón de CTR (5%). Sobre
    26,6M de filas eso está mal calibrado: el grupo sin perfil tiene 5,335% de CTR
    contra 5,132% del grupo con perfil —una razón de 1,040, por debajo del umbral—
    y sin embargo la diferencia tiene **z = 11,0**. Con ese umbral, las ocho
    columnas salían marcadas *"ausencia no informativa"*, contradiciendo el
    hallazgo de la Fase 1 de que el segmento sin perfil clickea por encima del
    promedio. Lo que decide es si la diferencia **existe**, y eso es una prueba de
    dos proporciones; el tamaño del efecto se reporta aparte para que el lector
    juzgue si además importa.
    """
    def decidir(r):
        p = r.pct_nulos_en_cruce / 100
        if p == 0:
            return "sin_accion", "no hay ausencia en el cruce"
        z = r.get("z")
        informativa = z is not None and abs(z) > z_critico
        evidencia = (f"CTR ausente/presente = {r.razon_ctr} con z = {z}"
                     if z is not None else "sin evidencia de z")
        if p <= config.UMBRAL_DESCARTE:
            return ("categoria_desconocido" if informativa else "descartar_fila"), (
                f"{r.pct_nulos_en_cruce}% <= 1%"
                + (f"; {evidencia} -> la ausencia ES un dato" if informativa else "")
            )
        if p <= config.UMBRAL_IMPUTACION:
            return ("categoria_desconocido" if informativa else "imputar_moda"), (
                f"{r.pct_nulos_en_cruce}% <= 5%"
                + (f"; {evidencia} -> la ausencia ES un dato" if informativa else "")
            )
        return "categoria_desconocido", (
            f"{r.pct_nulos_en_cruce}% > 5%; " + evidencia
            + (" -> la ausencia ES un dato" if informativa
               else " -> no distinguible del azar, pero el volumen impide imputar")
        )

    d = perfil.copy()
    d[["decision", "motivo"]] = d.apply(lambda r: pd.Series(decidir(r)), axis=1)
    return d


def construir(spark, decisiones: pd.DataFrame | None = None, escribir: bool = True):
    """El cruce, con el flag sin_perfil y el centinela aplicado a las categóricas."""
    imp, ads, usr = leer_bronze(spark)

    silver = (
        imp.join(F.broadcast(ads), on="adgroup_id", how="inner")
           .join(usr, on="userid", how="left")
           .withColumn("sin_perfil", F.col("cms_segid").isNull().cast("int"))
    )
    # Centinela en todas las categóricas de perfil. Si la decisión de una columna
    # fue imputar_moda o descartar_fila, el notebook lo hace explícito antes de aquí;
    # el centinela es el default porque es la decisión que la Fase 1 ya tomó para
    # las dos columnas con más ausencia.
    for c in CATEGORICAS_PERFIL:
        silver = silver.withColumn(c, F.coalesce(F.col(c).cast("int"), F.lit(CENTINELA)))

    silver = (
        silver
        # brand quedó FUERA como predictor: el cast fabricó 246.330 nulos que no
        # existen en el origen (Fase 1, sección 2). Solo sobrevive como flag.
        .withColumn("brand_conocida", F.col("brand").isNotNull().cast("int"))
        .withColumn("precio_valido", (F.col("price") > 0).cast("int"))
        .withColumn("dia_semana", F.dayofweek("ts_local"))
        .withColumn("es_finde", F.col("dia_semana").isin([1, 7]).cast("int"))
        .drop("brand", "nonclk", "ts_utc")
    )

    if escribir:
        (silver.write.mode("overwrite").option("compression", "zstd")
            .partitionBy("fecha_local").parquet(config.SILVER))
        silver = spark.read.parquet(config.SILVER)
    silver.createOrReplaceTempView("silver")
    return silver


def validar_cardinalidad(spark, n_bronze: int) -> pd.DataFrame:
    """El control de la Fase 1, repetido: un JOIN mal especificado infla las filas
    y el CTR resultante SIGUE pareciendo razonable. delta != 0 detiene el pipeline."""
    leer_bronze(spark)
    pruebas = []
    for etiqueta, sql in [
        ("impresiones ⋈ ad_feature (INNER)",
         "SELECT COUNT(*) FROM impresiones i JOIN ad_feature a ON i.adgroup_id = a.adgroup_id"),
        ("+ LEFT JOIN user_profile",
         """SELECT COUNT(*) FROM impresiones i
            JOIN ad_feature a ON i.adgroup_id = a.adgroup_id
            LEFT JOIN user_profile u ON i.userid = u.userid"""),
    ]:
        n = spark.sql(sql).collect()[0][0]
        pruebas.append({"prueba": etiqueta, "filas": n, "delta": n - n_bronze,
                        "veredicto": "OK" if n == n_bronze else
                                     ("DUPLICA" if n > n_bronze else "PIERDE FILAS")})
    for tabla, clave in (("ad_feature", "adgroup_id"), ("user_profile", "userid")):
        tot, dis = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT {clave}) FROM {tabla}").collect()[0]
        pruebas.append({"prueba": f"{tabla}.{clave} único", "filas": tot, "delta": tot - dis,
                        "veredicto": "OK" if tot == dis else "CLAVE NO ÚNICA"})
    card = pd.DataFrame(pruebas)
    assert (card.veredicto == "OK").all(), f"los joins alteran la cardinalidad:\n{card}"
    return card

In [ ]:
%%writefile adbd/gold.py
"""
GOLD · la tabla de entrenamiento y el registro de variables versionado (gold_ads_v1).

Este módulo es donde se gana o se pierde la Fase 2, porque es donde vive el riesgo
de FUGA DE INFORMACIÓN. La Fase 1 ya lo encontró una vez en W3 (`CURRENT ROW` en
lugar de `1 PRECEDING` metía el resultado a predecir dentro del feature y el split
temporal seguía viéndose impecable). Aquí la misma disciplina, aplicada a todo:

  Regla única: NINGÚN feature puede depender de una fila cuyo `clk` el modelo
  todavía no habría visto en el instante de la impresión.

De ahí salen las tres decisiones de diseño del módulo:

  1. Los CTR históricos por entidad se calculan con retardo de UN DÍA y ventana
     expansiva: para una impresión del día d, el histórico solo suma los días < d.
  2. Los features de usuario (fatiga) se calculan con ventana expansiva DENTRO del
     flujo, cerrada en la fila anterior (`rowsBetween(unboundedPreceding, -1)`), con
     el mismo desempate total que la Fase 1 necesitó para que W2 fuera reproducible.
  3. El CTR móvil del slot mantiene `RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING`,
     literalmente la consulta W3, porque ya estaba bien.

El costo de la regla es un día: el 06-may no se entrena, existe para que el 07-may
tenga historia. `verificar_fuga()` deja el control ejecutable.
"""
from __future__ import annotations

import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F

from . import config

# ---------------------------------------------------------------------------
# Registro de variables. Es un artefacto versionado: lo aceptado Y lo descartado
# con su motivo, que es lo que la Fase 1 prometió entregar en Gold.
# ---------------------------------------------------------------------------
CATEGORICAS = [
    "pid", "franja", "hora_local", "dia_semana",
    "final_gender_code", "age_level", "pvalue_level", "shopping_level",
    "occupation", "new_user_class_level", "cms_group_id", "bucket_gap",
]

NUMERICAS = (
    [f"ctr_hist_{e}" for e in config.ENTIDADES_HISTORICAS]
    + [f"log_imp_hist_{e}" for e in config.ENTIDADES_HISTORICAS]
    + [
        "ctr_usuario_previo", "log_imp_previas_usuario", "orden_impresion",
        "log_gap_seg", "ctr_movil_6h_slot",
        "log_price", "precio_rel_categoria",
        "sin_perfil", "brand_conocida", "precio_valido", "es_finde",
    ]
)

DESCARTADAS = {
    "brand": "el cast fabricó 246.330 nulos inexistentes en el origen (Fase 1 §2); "
             "usarla sería modelar un artefacto de la línea de carga. Sobrevive como "
             "flag brand_conocida.",
    "nonclk": "complemento exacto de clk dentro de la misma fila: fuga directa del objetivo.",
    "adgroup_id (histórico)": "846.811 entidades x 8 días = 6,8M de filas de historia contra "
                              "26,6M de impresiones; campaign_id lo contiene jerárquicamente.",
    "cms_segid": "97 niveles muy desbalanceados y redundante con cms_group_id, que es su "
                 "agregación declarada por la fuente.",
    "ts_local / time_stamp": "identificador temporal, no atributo: entra transformado "
                             "(hora_local, dia_semana, franja) y nunca en crudo.",
    "userid / adgroup_id (como categoría)": "cardinalidad 1,06M y 846k: no se codifican como "
                                            "categoría, entran a través de sus agregados históricos.",
}

OBJETIVO = "clk"
PROTEGIDOS = ["final_gender_code", "age_level", "new_user_class_level"]


# ---------------------------------------------------------------------------
# 1. Prior global diferido: el CTR de TODOS los días anteriores a d.
# ---------------------------------------------------------------------------
def _prior_diferido(silver):
    diario = silver.groupBy("fecha_local").agg(
        F.count(F.lit(1)).alias("imp_d"), F.sum(OBJETIVO).alias("clk_d")
    )
    w = Window.orderBy("fecha_local").rowsBetween(Window.unboundedPreceding, -1)
    return diario.select(
        "fecha_local",
        (F.sum("clk_d").over(w) / F.sum("imp_d").over(w)).alias("p0_diferido"),
    )


# ---------------------------------------------------------------------------
# 2. CTR histórico por entidad, con retardo de un día y suavizado de m-estimación.
#
#       ctr_hist = (clicks_previos + m * p0) / (impresiones_previas + m)
#
#    Con impresiones_previas = 0 (entidad nueva) el feature ES el prior: no hay
#    ningún caso en que el modelo reciba un NaN ni un cero engañoso.
# ---------------------------------------------------------------------------
def _ctr_historico(silver, prior, entidad: str, m: float):
    diario = silver.groupBy(entidad, "fecha_local").agg(
        F.count(F.lit(1)).alias("imp_d"), F.sum(OBJETIVO).alias("clk_d")
    )
    w = Window.partitionBy(entidad).orderBy("fecha_local").rowsBetween(
        Window.unboundedPreceding, -1
    )
    hist = diario.select(
        F.col(entidad),
        F.col("fecha_local"),
        F.coalesce(F.sum("imp_d").over(w), F.lit(0)).cast("double").alias("imp_prev"),
        F.coalesce(F.sum("clk_d").over(w), F.lit(0)).cast("double").alias("clk_prev"),
    )
    return (
        hist.join(prior, on="fecha_local", how="left")
        .select(
            F.col(entidad),
            F.col("fecha_local"),
            ((F.col("clk_prev") + F.lit(m) * F.col("p0_diferido"))
             / (F.col("imp_prev") + F.lit(m))).alias(f"ctr_hist_{entidad}"),
            F.log1p("imp_prev").alias(f"log_imp_hist_{entidad}"),
        )
    )


# ---------------------------------------------------------------------------
# 3. CTR móvil del slot: la consulta W3 de la Fase 1, sin cambios.
#    RANGE (no ROWS) y cota superior en 1 PRECEDING, que EXCLUYE la hora actual.
# ---------------------------------------------------------------------------
def _ctr_movil_slot(silver):
    ts_min = silver.agg(F.min("time_stamp")).collect()[0][0]
    base = (
        silver.withColumn("hora_abs", F.floor((F.col("time_stamp") - F.lit(ts_min)) / 3600).cast("bigint"))
        .groupBy("pid", "hora_abs")
        .agg(F.count(F.lit(1)).alias("imp_h"), F.sum(OBJETIVO).alias("clk_h"))
        .withColumn("ctr_hora", F.col("clk_h") / F.col("imp_h"))
    )
    w = Window.partitionBy("pid").orderBy("hora_abs").rangeBetween(-6, -1)
    return ts_min, base.select(
        "pid", "hora_abs", F.avg("ctr_hora").over(w).alias("ctr_movil_6h_slot")
    )


# ---------------------------------------------------------------------------
# 4. Precio relativo a su categoría. `price` no deriva del objetivo, así que no
#    hay fuga posible; aun así la referencia se calcula SOLO sobre la ventana de
#    entrenamiento, porque calcular estadísticas sobre el test es transducción y
#    en producción esa media no existiría todavía.
# ---------------------------------------------------------------------------
def _precio_referencia(silver):
    tr = silver.filter(
        (F.col("fecha_local") >= F.lit(config.TRAIN_INI))
        & (F.col("fecha_local") <= F.lit(config.TRAIN_FIN))
    )
    return tr.groupBy("cate_id").agg(F.avg("price").alias("precio_medio_cate"))


# ---------------------------------------------------------------------------
# Construcción completa
# ---------------------------------------------------------------------------
def construir(spark, silver, escribir: bool = True, m: float | None = None):
    """Devuelve la tabla Gold. Cada bloque está comentado con qué fuga evita."""
    m = config.M_SUAVIZADO if m is None else m
    prior = _prior_diferido(silver).cache()

    g = silver
    for e in config.ENTIDADES_HISTORICAS:
        h = _ctr_historico(silver, prior, e, m)
        # cate_id/pid/customer son tablas chicas; campaign_id es la única grande.
        g = g.join(F.broadcast(h) if e in ("pid", "cate_id") else h,
                   on=[e, "fecha_local"], how="left")

    # --- Fatiga del usuario. El desempate (time_stamp, adgroup_id, pid) es el que
    # la Fase 1 tuvo que agregar para que W2 fuera determinista: con 6.220.484 pares
    # usuario-segundo empatados, sin él la ventana da un resultado distinto por motor.
    w_user = Window.partitionBy("userid").orderBy("time_stamp", "adgroup_id", "pid")
    w_prev = w_user.rowsBetween(Window.unboundedPreceding, -1)
    g = (
        g.join(F.broadcast(prior), on="fecha_local", how="left")
        .withColumn("orden_impresion", F.row_number().over(w_user))
        .withColumn("gap_seg", F.col("time_stamp") - F.lag("time_stamp").over(w_user))
        .withColumn("clk_prev_u", F.coalesce(F.sum(OBJETIVO).over(w_prev), F.lit(0)).cast("double"))
    )
    mu = config.M_SUAVIZADO_USUARIO
    g = (
        g.withColumn("imp_prev_u", (F.col("orden_impresion") - 1).cast("double"))
        .withColumn(
            "ctr_usuario_previo",
            (F.col("clk_prev_u") + F.lit(mu) * F.col("p0_diferido"))
            / (F.col("imp_prev_u") + F.lit(mu)),
        )
        .withColumn("log_imp_previas_usuario", F.log1p("imp_prev_u"))
        .withColumn(
            "bucket_gap",
            F.when(F.col("gap_seg").isNull(), "primera")
            .when(F.col("gap_seg") <= 300, "0-5min")
            .when(F.col("gap_seg") <= 3600, "5-60min")
            .when(F.col("gap_seg") <= 86400, "1-24h")
            .otherwise("mas_1d"),
        )
        .withColumn("log_gap_seg", F.log1p(F.coalesce(F.col("gap_seg"), F.lit(0.0))))
        # El tope en 20 replica exactamente el LEAST(orden_impresion, 20) de W2:
        # más allá la curva de CTR es plana y la cola larga solo agrega varianza.
        .withColumn("orden_impresion", F.least(F.col("orden_impresion"), F.lit(20)).cast("double"))
    )

    ts_min, movil = _ctr_movil_slot(silver)
    g = (
        g.withColumn("hora_abs", F.floor((F.col("time_stamp") - F.lit(ts_min)) / 3600).cast("bigint"))
        .join(F.broadcast(movil), on=["pid", "hora_abs"], how="left")
    )

    g = (
        g.join(F.broadcast(_precio_referencia(silver)), on="cate_id", how="left")
        .withColumn("log_price", F.log1p(F.greatest(F.col("price"), F.lit(0.0))))
        .withColumn(
            "precio_rel_categoria",
            F.col("price") / F.when(F.col("precio_medio_cate") > 0,
                                    F.col("precio_medio_cate")).otherwise(F.lit(None)),
        )
    )

    # Relleno FINAL. Un null que llega al VectorAssembler aborta el ajuste, y lo
    # correcto es que cada relleno tenga una razón, no un 0 por defecto:
    #   ctr_movil_6h_slot  -> p0 diferido (primeras horas del período, sin historia)
    #   precio_rel_categoria -> 1.0 (categoría sin referencia = precio "normal")
    g = (
        g.withColumn("ctr_movil_6h_slot",
                     F.coalesce(F.col("ctr_movil_6h_slot"), F.col("p0_diferido")))
        .withColumn("precio_rel_categoria",
                    F.coalesce(F.col("precio_rel_categoria"), F.lit(1.0)))
        .withColumn("hora_local", F.col("hora_local").cast("int"))
    )

    # Identificadores. NO son features (ver DESCARTADAS): viajan en la tabla porque
    # el experimento de recomendación necesita la pareja (userid, cate_id) y porque
    # sin adgroup_id no se puede auditar una predicción hasta el aviso que la produjo.
    columnas = (["userid", "adgroup_id", "cate_id", "time_stamp", "fecha_local", OBJETIVO]
                + CATEGORICAS + NUMERICAS)
    columnas = list(dict.fromkeys(columnas))
    gold = g.select(*columnas)

    if escribir:
        (gold.write.mode("overwrite").option("compression", "zstd")
            .partitionBy("fecha_local").parquet(config.GOLD))
        gold = spark.read.parquet(config.GOLD)
    gold.createOrReplaceTempView("gold")
    return gold


# ---------------------------------------------------------------------------
# Split temporal
# ---------------------------------------------------------------------------
def particionar(gold):
    """(train, val, test). El burn-in NO se devuelve: existe solo como historia."""
    f = F.col("fecha_local")
    train = gold.filter((f >= F.lit(config.TRAIN_INI)) & (f <= F.lit(config.TRAIN_FIN)))
    val = gold.filter(f == F.lit(config.DIA_VAL))
    test = gold.filter(f == F.lit(config.DIA_TEST))
    return train, val, test


def resumen_split(gold) -> pd.DataFrame:
    """Filas, positivos y CTR por día, con la etiqueta del split. El CTR por día
    tiene que ser estable: si el día de test tuviera un CTR muy distinto, la
    comparación de métricas entre val y test no significaría nada."""
    d = (
        gold.groupBy("fecha_local")
        .agg(F.count(F.lit(1)).alias("filas"), F.sum(OBJETIVO).alias("positivos"))
        .orderBy("fecha_local")
        .toPandas()
    )
    d["fecha_local"] = d["fecha_local"].astype(str)
    d["ctr_pct"] = (d.positivos / d.filas * 100).round(3)

    def etiqueta(f):
        if f == config.DIA_BURNIN:
            return "burn-in (no se entrena)"
        if config.TRAIN_INI <= f <= config.TRAIN_FIN:
            return "train"
        if f == config.DIA_VAL:
            return "val"
        if f == config.DIA_TEST:
            return "test"
        return "fuera de rango"

    d["split"] = d.fecha_local.map(etiqueta)
    return d


# ---------------------------------------------------------------------------
# Control ejecutable de fuga
# ---------------------------------------------------------------------------
def verificar_fuga(gold, spark) -> pd.DataFrame:
    """Tres pruebas que tienen que pasar para que las métricas signifiquen algo.

    1. Ninguna correlación |r| > 0.95 entre un feature numérico y el objetivo.
       Un feature que casi ES el objetivo es fuga, no señal.
    2. El primer día entrenable NO tiene históricos nulos (el burn-in cumplió su función).
    3. Los CTR históricos del día d están dentro del rango plausible de una tasa
       (0,1] y no coinciden con el CTR observado del propio día d, que sería el
       síntoma de que la ventana quedó abierta en CURRENT ROW.
    """
    filas = []

    objetivo_num = [c for c in NUMERICAS if c not in ("sin_perfil", "brand_conocida",
                                                      "precio_valido", "es_finde")]
    corr = {c: gold.stat.corr(c, OBJETIVO) for c in objetivo_num}
    peor = max(corr, key=lambda c: abs(corr[c] or 0))
    filas.append({
        "prueba": "correlación feature-objetivo",
        "detalle": f"máx |r| = {abs(corr[peor] or 0):.4f} en {peor}",
        "veredicto": "OK" if abs(corr[peor] or 0) <= 0.95 else "FUGA",
    })

    n_nulos = gold.filter(F.col("fecha_local") == F.lit(config.TRAIN_INI)).select(
        F.sum(
            sum(
                (F.col(f"ctr_hist_{e}").isNull()).cast("int")
                for e in config.ENTIDADES_HISTORICAS
            )
        ).alias("n")
    ).collect()[0]["n"] or 0
    filas.append({
        "prueba": "históricos disponibles el primer día entrenable",
        "detalle": f"{int(n_nulos)} nulos el {config.TRAIN_INI}",
        "veredicto": "OK" if n_nulos == 0 else "BURN-IN INSUFICIENTE",
    })

    rangos = gold.select(
        *[F.min(f"ctr_hist_{e}").alias(f"min_{e}") for e in config.ENTIDADES_HISTORICAS],
        *[F.max(f"ctr_hist_{e}").alias(f"max_{e}") for e in config.ENTIDADES_HISTORICAS],
    ).collect()[0].asDict()
    fuera = {k: v for k, v in rangos.items() if v is not None and not (0 < v <= 1)}
    filas.append({
        "prueba": "CTR históricos dentro de (0, 1]",
        "detalle": "todos dentro del rango" if not fuera else str(fuera),
        "veredicto": "OK" if not fuera else "VALOR IMPOSIBLE",
    })

    return pd.DataFrame(filas)


def registro_variables() -> pd.DataFrame:
    """El artefacto versionado que prometió la Fase 1: aceptadas y descartadas."""
    filas = [{"variable": c, "tipo": "categórica", "estado": "aceptada", "motivo": ""}
             for c in CATEGORICAS]
    filas += [{"variable": c, "tipo": "numérica", "estado": "aceptada", "motivo": ""}
              for c in NUMERICAS]
    filas += [{"variable": k, "tipo": "-", "estado": "DESCARTADA", "motivo": v}
              for k, v in DESCARTADAS.items()]
    return pd.DataFrame(filas)

In [ ]:
%%writefile adbd/transformadores.py
"""
Transformers propios. Dos, y cada uno existe porque MLlib no trae el equivalente.

`CodificadorCentinela` es higiene: un null que llega a StringIndexer con
handleInvalid="error" aborta el ajuste, y con "skip" BORRA la fila en silencio —
que es justo el fallo de calidad que la Fase 1 persiguió en la línea de carga.
Aquí el null se convierte en una categoría explícita antes de indexar.

`CorrectorPrior` es la pieza analítica del módulo. Al submuestrear negativos, el
modelo aprende sobre una prevalencia inventada (35% en vez de 5,14%) y sus
probabilidades quedan sistemáticamente infladas. El orden de las predicciones no
cambia —por eso AUC-ROC sobrevive intacto—, pero cualquier decisión que compare la
probabilidad contra un umbral, y cualquier métrica de calibración (LogLoss, Brier),
quedan mal. La corrección es exacta y tiene una línea:

    odds_reales = odds_submuestreadas · r        con r = tasa de negativos conservada
    p           = p_s·r / (p_s·r + 1 − p_s)

Comprobación: r = 1 devuelve p = p_s; r → 0 devuelve p → 0. Ambas se prueban en
tests/test_humo.py, porque una fórmula de calibración mal puesta no falla, miente.
"""
from __future__ import annotations

from pyspark import keyword_only
from pyspark.ml import Transformer
from pyspark.ml.functions import vector_to_array
from pyspark.ml.param import Param, Params, TypeConverters
from pyspark.ml.param.shared import HasInputCol, HasInputCols, HasOutputCol, HasOutputCols
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.sql import functions as F


class CodificadorCentinela(
    Transformer, HasInputCols, HasOutputCols, DefaultParamsReadable, DefaultParamsWritable
):
    """Castea a string y reemplaza nulls por un centinela explícito."""

    centinela = Param(
        Params._dummy(), "centinela", "token para el valor ausente",
        typeConverter=TypeConverters.toString,
    )

    @keyword_only
    def __init__(self, inputCols=None, outputCols=None, centinela="DESCONOCIDO"):
        super().__init__()
        self._setDefault(centinela="DESCONOCIDO")
        self.setParams(**self._input_kwargs)

    @keyword_only
    def setParams(self, inputCols=None, outputCols=None, centinela="DESCONOCIDO"):
        return self._set(**self._input_kwargs)

    def getCentinela(self):
        return self.getOrDefault(self.centinela)

    def _transform(self, df):
        entradas = self.getInputCols()
        salidas = self.getOutputCols() or [f"{c}_cat" for c in entradas]
        token = self.getCentinela()
        for ent, sal in zip(entradas, salidas):
            df = df.withColumn(
                sal, F.coalesce(F.col(ent).cast("string"), F.lit(token))
            )
        return df


class CorrectorPrior(
    Transformer, HasInputCol, HasOutputCol, DefaultParamsReadable, DefaultParamsWritable
):
    """Devuelve la probabilidad a la prevalencia real tras submuestrear negativos.

    inputCol  : columna `probability` (Vector) o una columna double con p(clk=1)
    outputCol : probabilidad recalibrada (double)
    """

    tasaNegativos = Param(
        Params._dummy(), "tasaNegativos",
        "fracción de negativos conservada en el entrenamiento (r en (0, 1])",
        typeConverter=TypeConverters.toFloat,
    )

    @keyword_only
    def __init__(self, inputCol="probability", outputCol="p_calibrada", tasaNegativos=1.0):
        super().__init__()
        self._setDefault(inputCol="probability", outputCol="p_calibrada", tasaNegativos=1.0)
        self.setParams(**self._input_kwargs)

    @keyword_only
    def setParams(self, inputCol="probability", outputCol="p_calibrada", tasaNegativos=1.0):
        return self._set(**self._input_kwargs)

    def getTasaNegativos(self):
        return self.getOrDefault(self.tasaNegativos)

    def _transform(self, df):
        ent, sal = self.getInputCol(), self.getOutputCol()
        r = float(self.getTasaNegativos())
        if not 0 < r <= 1:
            raise ValueError(f"tasaNegativos tiene que estar en (0, 1]; llegó {r}")
        tipo = dict(df.dtypes)[ent]
        # `probability` de MLlib es un Vector; una columna ya escalar entra tal cual.
        ps = (F.col(ent).cast("double") if tipo in ("double", "float")
              else vector_to_array(F.col(ent))[1])
        return df.withColumn(
            sal, (ps * F.lit(r)) / (ps * F.lit(r) + (F.lit(1.0) - ps))
        )


def recalibrar(p_submuestreada: float, tasa_negativos: float) -> float:
    """Misma fórmula, en Python puro, para poder probarla sin levantar Spark."""
    r = tasa_negativos
    return (p_submuestreada * r) / (p_submuestreada * r + (1.0 - p_submuestreada))

In [ ]:
%%writefile adbd/features.py
"""
Ensamblado del Pipeline de MLlib: Transformers y Estimators, en ese orden.

    CodificadorCentinela  (Transformer propio)   null -> categoría explícita
    StringIndexer         (Estimator)            categoría -> índice, por columna
    OneHotEncoder         (Estimator)            índice -> vector disperso
    VectorAssembler       (Transformer)          todo -> un solo vector
    StandardScaler        (Estimator)            centrado/escala, solo si el modelo lo necesita

Dos decisiones que el informe defiende:

1. **Nada de alta cardinalidad entra codificado.** adgroup_id (846.811), campaign_id
   (423k) y userid (1,06M) NO se indexan: un OneHotEncoder sobre ellos produce un
   vector de millones de columnas y un StringIndexer sobre 1,06M de niveles es un
   `collect` del vocabulario al driver. Entran por sus agregados históricos
   diferidos (gold.py), que es justo la representación que sí generaliza a una
   entidad nueva.

2. **`handleInvalid="keep"` en ambos.** Una categoría que aparece en test y no en
   train existe de verdad (campañas nuevas todos los días) y tiene que recibir un
   índice, no tumbar la predicción ni desaparecer de la evaluación.

El escalado es condicional: la regresión logística con regularización lo necesita,
los árboles no lo usan y pagarlo sería tiempo regalado.
"""
from __future__ import annotations

from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler

from . import gold as gold_mod
from .transformadores import CodificadorCentinela


def etapas_features(categoricas=None, numericas=None, escalar: bool = False,
                    salida: str = "features"):
    """Las etapas de preparación, sin el estimador final."""
    categoricas = categoricas or gold_mod.CATEGORICAS
    numericas = numericas or gold_mod.NUMERICAS

    cent = [f"{c}_cat" for c in categoricas]
    idx = [f"{c}_idx" for c in categoricas]
    ohe = [f"{c}_ohe" for c in categoricas]

    etapas = [
        CodificadorCentinela(inputCols=categoricas, outputCols=cent),
        StringIndexer(inputCols=cent, outputCols=idx, handleInvalid="keep",
                      stringOrderType="frequencyDesc"),
        OneHotEncoder(inputCols=idx, outputCols=ohe, handleInvalid="keep", dropLast=True),
    ]
    destino = "features_sin_escalar" if escalar else salida
    etapas.append(
        VectorAssembler(inputCols=ohe + list(numericas), outputCol=destino,
                        handleInvalid="error")
    )
    if escalar:
        etapas.append(
            StandardScaler(inputCol=destino, outputCol=salida,
                           withMean=False, withStd=True)
        )
    return etapas


def pipeline(estimador, categoricas=None, numericas=None, escalar: bool = False):
    return Pipeline(stages=etapas_features(categoricas, numericas, escalar) + [estimador])


def nombres_features(modelo_pipeline, categoricas=None, numericas=None) -> list[str]:
    """Nombres en el MISMO orden que el vector ensamblado, para poder leer los
    coeficientes y las importancias. Sin esto la interpretación es adivinanza.

    El ancho de cada bloque one-hot NO se deduce: se lee de `categorySizes` del
    OneHotEncoderModel ajustado, que es la única fuente que conoce el efecto
    combinado de dropLast y handleInvalid="keep". Deducirlo a mano desalinea los
    nombres por una posición y hace que el informe atribuya un coeficiente a la
    variable equivocada — un error que no se ve porque no falla.
    """
    categoricas = categoricas or gold_mod.CATEGORICAS
    numericas = numericas or gold_mod.NUMERICAS
    indexador = next(e for e in modelo_pipeline.stages
                     if type(e).__name__ == "StringIndexerModel")
    codificador = next(e for e in modelo_pipeline.stages
                       if type(e).__name__ == "OneHotEncoderModel")
    tamanos = list(codificador.categorySizes)
    dropea = bool(codificador.getDropLast())

    nombres = []
    for col, labels, n_cat in zip(categoricas, indexador.labelsArray, tamanos):
        etiquetas = [str(l) for l in labels] + ["__desconocido__"]
        ancho = n_cat - 1 if dropea else n_cat
        etiquetas = (etiquetas + [f"__nivel_{i}__" for i in range(ancho)])[:ancho]
        nombres += [f"{col}={l}" for l in etiquetas]
    return nombres + list(numericas)


def importancias(modelo_pipeline, categoricas=None, numericas=None, top: int = 25):
    """Coeficientes (modelos lineales) o importancias (árboles), con nombre.

    Sirve para dos cosas distintas y las dos importan en la defensa: explicar qué
    mueve la predicción, y detectar fuga que las métricas no delatan. Una variable
    que concentra casi toda la importancia suele ser una fuga, no un hallazgo.
    """
    import pandas as pd

    estimador = modelo_pipeline.stages[-1]
    nombres = nombres_features(modelo_pipeline, categoricas, numericas)
    if hasattr(estimador, "coefficients"):
        pesos = list(estimador.coefficients.toArray())
        clase = "coeficiente"
    elif hasattr(estimador, "featureImportances"):
        pesos = list(estimador.featureImportances.toArray())
        clase = "importancia"
    else:
        return pd.DataFrame()
    n = min(len(nombres), len(pesos))
    d = pd.DataFrame({"variable": nombres[:n], clase: pesos[:n]})
    d["magnitud"] = d[clase].abs()
    d["peso_relativo_pct"] = (d.magnitud / d.magnitud.sum() * 100).round(2)
    return d.sort_values("magnitud", ascending=False).head(top).reset_index(drop=True)

In [ ]:
%%writefile adbd/modelos.py
"""
Modelamiento distribuido. Cuatro experimentos supervisados y uno de recomendación.

El desbalance (CTR 5,19% en la ventana de entrenamiento, ~1:18) se ataca por
SUBMUESTREO DE NEGATIVOS, no por sobremuestreo ni por `weightCol` a secas, y la
razón es de costo medido: el train son 16.744.897 filas y las 12 combinaciones que
suman las grillas de los cuatro experimentos, con `CrossValidator` de 3 pliegues,
son 36 ajustes completos más los cuatro reajustes finales. El submuestreo baja el
entrenamiento a 2.458.575 filas conservando TODOS los positivos.

Lo que el submuestreo cuesta se paga y se declara:
  · la probabilidad queda inflada  -> se recalibra (transformadores.CorrectorPrior);
  · la evaluación se hace SIEMPRE sobre el conjunto completo sin submuestrear,
    porque AUC-PR depende de la prevalencia y medirlo sobre la muestra daría un
    número bonito que no existe en producción.

El experimento `lr_pesos_completo` es la contraprueba, y en la corrida sobre el
dataset real dio el veredicto más limpio posible: **AUC-PR 0,093432 con weightCol
sobre las 16,7M de filas contra 0,093534 con submuestreo sobre 2,46M** — una
diferencia de −0,0001, dentro del ruido— y **197,1 s contra 43,0 s, 4,6x el
tiempo**. El atajo no cuesta precisión; compra un factor 4,6 de cómputo.
"""
from __future__ import annotations

import time

from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F

from . import config, features as feat, gold as gold_mod


# ---------------------------------------------------------------------------
# Desbalance
# ---------------------------------------------------------------------------
def submuestrear_negativos(df, tasa: float | None = None, semilla: int | None = None):
    """Conserva TODOS los positivos y una fracción `tasa` de los negativos.

    Devuelve (df_submuestreado, info). `info["tasa_efectiva"]` es la fracción de
    negativos realmente conservada, que no es exactamente `tasa` porque
    `sample` es de Bernoulli por fila. La recalibración usa la efectiva, no la
    pedida: con la nominal la corrección queda sesgada en el tercer decimal.
    """
    tasa = config.TASA_NEGATIVOS if tasa is None else tasa
    semilla = config.SEMILLA if semilla is None else semilla
    pos = df.filter(F.col(gold_mod.OBJETIVO) == 1)
    neg = df.filter(F.col(gold_mod.OBJETIVO) == 0)
    n_neg = neg.count()
    neg_s = neg.sample(withReplacement=False, fraction=tasa, seed=semilla)
    n_neg_s = neg_s.count()
    n_pos = pos.count()
    muestra = pos.unionByName(neg_s)
    info = {
        "tasa_nominal": tasa,
        "tasa_efectiva": n_neg_s / n_neg if n_neg else 1.0,
        "positivos": n_pos,
        "negativos_originales": n_neg,
        "negativos_conservados": n_neg_s,
        "filas_entrenamiento": n_pos + n_neg_s,
        "prevalencia_muestra": n_pos / (n_pos + n_neg_s),
        "prevalencia_real": n_pos / (n_pos + n_neg),
    }
    return muestra, info


# ---------------------------------------------------------------------------
# Definición de los experimentos
# ---------------------------------------------------------------------------
def catalogo_experimentos(numericas_reducidas=None):
    """Cada experimento responde UNA pregunta. Un barrido de modelos sin pregunta
    produce una tabla que no se puede defender ante el panel."""
    lr = LogisticRegression(labelCol=gold_mod.OBJETIVO, featuresCol="features",
                            maxIter=50, family="binomial")
    gbt = GBTClassifier(labelCol=gold_mod.OBJETIVO, featuresCol="features",
                        maxIter=40, maxDepth=5, stepSize=0.1, seed=config.SEMILLA)
    rf = RandomForestClassifier(labelCol=gold_mod.OBJETIVO, featuresCol="features",
                                numTrees=60, maxDepth=8, seed=config.SEMILLA,
                                subsamplingRate=0.7)

    return {
        "lr_contexto": {
            "pregunta": "¿Cuánto se predice SIN historia, solo con perfil y contexto?",
            "estimador": lr,
            "escalar": True,
            "numericas": numericas_reducidas,
            "grid": (ParamGridBuilder()
                     .addGrid(lr.regParam, [0.0, 0.01])
                     .addGrid(lr.elasticNetParam, [0.0])
                     .build()),
        },
        "lr_completo": {
            "pregunta": "¿Cuánto agregan los CTR históricos diferidos y la fatiga?",
            "estimador": lr,
            "escalar": True,
            "numericas": None,
            "grid": (ParamGridBuilder()
                     .addGrid(lr.regParam, [0.0, 0.001, 0.01])
                     .addGrid(lr.elasticNetParam, [0.0, 0.5])
                     .build()),
        },
        "rf_completo": {
            "pregunta": "¿Gana un ensemble de árboles sin interacciones explícitas?",
            "estimador": rf,
            "escalar": False,
            "numericas": None,
            "grid": (ParamGridBuilder()
                     .addGrid(rf.maxDepth, [6, 10])
                     .addGrid(rf.numTrees, [60])
                     .build()),
        },
        "gbt_completo": {
            "pregunta": "¿Compensa el boosting su costo de cómputo en AUC-PR?",
            "estimador": gbt,
            "escalar": False,
            "numericas": None,
            "grid": (ParamGridBuilder()
                     .addGrid(gbt.maxDepth, [4, 6])
                     .addGrid(gbt.maxIter, [40])
                     .build()),
        },
    }


# ---------------------------------------------------------------------------
# Tuning
# ---------------------------------------------------------------------------
def ajustar_con_cv(train_ds, spec: dict, folds: int = 3,
                   fraccion_busqueda: float | None = None, metrica: str = "areaUnderPR"):
    """CrossValidator + ParamGridBuilder sobre la ventana de entrenamiento.

    Presupuesto de cómputo, declarado: la BÚSQUEDA corre sobre una submuestra
    aleatoria de `fraccion_busqueda` del train ya submuestreado; el modelo GANADOR
    se reajusta sobre el train submuestreado completo. Buscar sobre el 100% sería
    `folds x |grid|` ajustes completos y no cabe en la sesión.

    Sobre el k-fold aleatorio: parte la ventana de entrenamiento sin respetar el
    orden temporal, lo que normalmente sería una fuga. Aquí NO lo es, porque la
    protección temporal vive en los FEATURES (gold.py los construye con retardo),
    no en el corte: una fila del 10-may no contiene nada del 11-may aunque caiga
    en el mismo pliegue. La afirmación no se deja en palabras: `brecha_optimismo`
    compara la métrica de CV contra la del día de validación retenido, y esa
    diferencia es lo que el informe reporta.
    """
    fraccion_busqueda = config.FRACCION_BUSQUEDA if fraccion_busqueda is None else fraccion_busqueda
    pipe = feat.pipeline(spec["estimador"], numericas=spec.get("numericas"),
                         escalar=spec.get("escalar", False))
    ev = BinaryClassificationEvaluator(labelCol=gold_mod.OBJETIVO,
                                       rawPredictionCol="rawPrediction",
                                       metricName=metrica)
    cv = CrossValidator(estimator=pipe, estimatorParamMaps=spec["grid"], evaluator=ev,
                        numFolds=folds, parallelism=1, seed=config.SEMILLA,
                        collectSubModels=False)

    busqueda = (train_ds.sample(False, fraccion_busqueda, seed=config.SEMILLA)
                if fraccion_busqueda < 1 else train_ds)
    t0 = time.perf_counter()
    cvm = cv.fit(busqueda)
    t_busqueda = round(time.perf_counter() - t0, 1)

    mejor_idx = int(max(range(len(cvm.avgMetrics)), key=lambda i: cvm.avgMetrics[i]))
    mejores = {p.name: v for p, v in spec["grid"][mejor_idx].items()}

    t0 = time.perf_counter()
    modelo = pipe.copy(spec["grid"][mejor_idx]).fit(train_ds)
    t_ajuste = round(time.perf_counter() - t0, 1)

    info = {
        "n_combinaciones": len(spec["grid"]),
        "folds": folds,
        "fraccion_busqueda": fraccion_busqueda,
        "filas_busqueda": busqueda.count(),
        "metrica_cv": metrica,
        "cv_mejor": float(cvm.avgMetrics[mejor_idx]),
        "cv_todas": [float(m) for m in cvm.avgMetrics],
        "mejores_parametros": {k: (float(v) if isinstance(v, (int, float)) else str(v))
                               for k, v in mejores.items()},
        "t_busqueda_s": t_busqueda,
        "t_ajuste_final_s": t_ajuste,
        "ajustes_realizados": folds * len(spec["grid"]) + 1,
    }
    return modelo, info


# ---------------------------------------------------------------------------
# Línea base honesta
# ---------------------------------------------------------------------------
def agregar_pesos(df, col: str = "peso"):
    """Pesos balanceados para el experimento de contraste: el mismo modelo sobre
    el train COMPLETO, sin submuestrear, con `weightCol`. Es la contraprueba de
    que el submuestreo es un atajo de costo y no un truco para mejorar la métrica."""
    n = df.count()
    n_pos = df.filter(F.col(gold_mod.OBJETIVO) == 1).count()
    w_pos = (n - n_pos) / n_pos if n_pos else 1.0
    info = {"filas": n, "positivos": n_pos, "peso_positivo": round(w_pos, 3),
            "peso_negativo": 1.0}
    return df.withColumn(
        col, F.when(F.col(gold_mod.OBJETIVO) == 1, F.lit(float(w_pos))).otherwise(F.lit(1.0))
    ), info


def curva_aprendizaje(train_ds, spec, fracciones=(0.1, 0.25, 0.5, 1.0), evaluar=None,
                      corrector=None, metrica_fn=None):
    """¿Más datos o más modelo? Ajusta el mismo pipeline sobre fracciones
    crecientes del entrenamiento y registra AUC-PR y tiempo.

    Es la respuesta cuantitativa a la pregunta que la Fase 1 dejó abierta: si la
    curva ya está plana al 100% del dataset actual, incorporar `behavior_log`
    (26x el volumen) compra tiempo de cómputo y no precisión, y esa es una
    decisión de escalamiento defendible con un número en vez de una intuición.

    Resultado sobre el dataset real: de 245.848 a 2.458.575 filas —10x— el AUC-PR
    pasa de 0,091259 a 0,091470, **+0,23%**. La curva está plana, y la conclusión
    sobre `behavior_log` se sostiene en ese número. Ojo con lo que NO dice: está
    plana *para este espacio de features y este modelo*. Más volumen serviría si
    trajera features nuevas (el comportamiento de navegación que `behavior_log`
    contiene), no más filas de las mismas.
    """
    import pandas as pd

    pipe = feat.pipeline(spec["estimador"], numericas=spec.get("numericas"),
                         escalar=spec.get("escalar", False))
    filas = []
    for f in fracciones:
        sub = train_ds.sample(False, f, seed=config.SEMILLA) if f < 1 else train_ds
        n = sub.count()
        t0 = time.perf_counter()
        modelo = pipe.fit(sub)
        t = round(time.perf_counter() - t0, 1)
        pred = modelo.transform(evaluar)
        if corrector is not None:
            pred = corrector.transform(pred)
        m = metrica_fn(pred)
        filas.append({"fraccion": f, "filas": n, "t_ajuste_s": t,
                      "auc_pr": m["auc_pr"], "auc_roc": m["auc_roc"],
                      "filas_por_segundo": round(n / t) if t else None})
        print(f"  {f:>5.0%} · {n:>10,} filas · {t:>7.1f} s · AUC-PR {m['auc_pr']:.5f}")
    return pd.DataFrame(filas)


def linea_base_prior(train, evaluar):
    """Predice el CTR histórico constante para todas las impresiones.

    No es relleno: fija el piso real. Con la prevalencia de 5,03% del día de test,
    un clasificador que acierta el 94,97% de las veces diciendo siempre "no click"
    tiene accuracy excelente y AUC-PR igual a la prevalencia. Sin esta línea base,
    el AUC-PR de 0,0935 del mejor modelo parecería malo; contra el piso de 0,0503
    es **1,86x**, y eso es lo que se defiende.
    """
    p0 = train.agg(F.avg(gold_mod.OBJETIVO)).collect()[0][0]
    return p0, evaluar.withColumn("p_calibrada", F.lit(float(p0)))


# ---------------------------------------------------------------------------
# ALS · feedback implícito usuario x categoría
# ---------------------------------------------------------------------------
def matriz_implicita(df, col_item: str = "cate_id"):
    """clicks por (usuario, categoría) en la ventana dada. Solo clicks: una
    impresión sin click no es una señal negativa, es ausencia de evidencia —
    exactamente el supuesto que `implicitPrefs=True` modela."""
    return (df.filter(F.col(gold_mod.OBJETIVO) == 1)
              .groupBy("userid", col_item)
              .agg(F.count(F.lit(1)).cast("double").alias("clicks")))


def verificar_ids_als(*matrices) -> dict:
    """ALS exige enteros de 32 bits. `userid` y `cate_id` son bigint; que quepan
    es plausible pero no se supone: se mide el máximo y se aborta si no cabe.
    Si algún día no cupiera, la salida es indexar, no truncar en silencio."""
    LIMITE = 2_147_483_647
    maximos = {}
    for m in matrices:
        r = m.agg(F.max("userid").alias("u"), F.max("cate_id").alias("i")).collect()[0]
        maximos["userid"] = max(maximos.get("userid", 0), int(r["u"] or 0))
        maximos["cate_id"] = max(maximos.get("cate_id", 0), int(r["i"] or 0))
    fuera = {k: v for k, v in maximos.items() if v > LIMITE}
    assert not fuera, f"no caben en int32 y habría que indexar: {fuera}"
    return maximos


def entrenar_als(train_mat, rank=None, reg=None, alpha=None, iters=None):
    train_mat = train_mat.withColumn("userid", F.col("userid").cast("int")) \
                         .withColumn("cate_id", F.col("cate_id").cast("int"))
    als = ALS(
        userCol="userid", itemCol="cate_id", ratingCol="clicks",
        rank=rank or config.ALS_RANK,
        regParam=reg or config.ALS_REG,
        alpha=alpha or config.ALS_ALPHA,
        maxIter=iters or config.ALS_ITER,
        implicitPrefs=True, coldStartStrategy="drop",
        nonnegative=True, seed=config.SEMILLA,
    )
    t0 = time.perf_counter()
    modelo = als.fit(train_mat)
    return modelo, round(time.perf_counter() - t0, 1)


def recomendaciones_populares(train_mat, k: int = None):
    """Línea base de recomendación: las k categorías más clickeadas, iguales para
    todos. Sin ella, un MAP@10 de 0,175 no se puede leer.

    Y en esta corrida fue la que dio el veredicto: la popularidad **gana**
    (MAP@10 0,2194 contra 0,1755 de ALS, NDCG 0,2753 contra 0,2150) y además
    cubre al 100% de los usuarios del test contra el 55,05% de ALS. Es un
    resultado negativo legítimo y se reporta como tal: a nivel de `cate_id` la
    granularidad es demasiado gruesa para que la personalización pague."""
    k = k or config.TOP_K
    top = [r["cate_id"] for r in
           train_mat.groupBy("cate_id").agg(F.sum("clicks").alias("c"))
                    .orderBy(F.col("c").desc()).limit(k).collect()]
    return top

In [ ]:
%%writefile adbd/evaluacion.py
"""
Evaluación predictiva. Todo se mide sobre el conjunto COMPLETO sin submuestrear.

Por qué AUC-PR y no accuracy: con 5,14% de positivos, predecir siempre "no click"
da 94,86% de accuracy y cero valor. AUC-PR se mueve con la prevalencia, así que
medirlo sobre una muestra con 35% de positivos daría un número que no existe en
producción. AUC-ROC sí es invariante al submuestreo de negativos (es puro orden),
y esa diferencia entre las dos métricas se reporta explícitamente porque es la
prueba de que la recalibración se hizo bien.

Y una métrica de negocio, porque el panel de la defensa no compra un AUC: el
**lift del decil superior**. En una plataforma de display no se decide "click o
no click", se ordena inventario; lo que importa es cuánto mejor rinde el 10% de
impresiones que el modelo pone arriba comparado con servir al azar.
"""
from __future__ import annotations

import math

import pandas as pd
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

from . import config
from .gold import OBJETIVO

EPS = 1e-12


def metricas_binarias(pred, col_p: str = "p_calibrada") -> dict:
    """AUC-ROC, AUC-PR, LogLoss, Brier y el CTR medio predicho vs el observado.

    Las dos últimas columnas son el control de calibración: si el CTR medio
    predicho no se parece al observado, la probabilidad no es una probabilidad
    y cualquier umbral de negocio construido sobre ella está mal puesto.
    """
    p = F.col(col_p)
    y = F.col(OBJETIVO).cast("double")
    pc = F.least(F.greatest(p, F.lit(EPS)), F.lit(1 - EPS))

    agg = pred.select(
        F.count(F.lit(1)).alias("n"),
        F.avg(y).alias("ctr_observado"),
        F.avg(p).alias("ctr_predicho"),
        F.avg(-(y * F.log(pc) + (1 - y) * F.log(1 - pc))).alias("logloss"),
        F.avg(F.pow(p - y, 2)).alias("brier"),
    ).collect()[0].asDict()

    ev = BinaryClassificationEvaluator(labelCol=OBJETIVO, rawPredictionCol=col_p)
    agg["auc_roc"] = ev.setMetricName("areaUnderROC").evaluate(pred)
    agg["auc_pr"] = ev.setMetricName("areaUnderPR").evaluate(pred)
    agg["lift_auc_pr"] = agg["auc_pr"] / agg["ctr_observado"] if agg["ctr_observado"] else None
    agg["sesgo_calibracion"] = (
        agg["ctr_predicho"] / agg["ctr_observado"] if agg["ctr_observado"] else None
    )
    return {k: (round(v, 6) if isinstance(v, float) else v) for k, v in agg.items()}


def tabla_deciles(pred, col_p: str = "p_calibrada", n: int = 10) -> pd.DataFrame:
    """CTR observado por decil de score. Es la traducción a negocio del AUC.

    Los cortes se obtienen con approxQuantile (distribuido) y no con NTILE: una
    ventana sin PARTITION BY sobre millones de filas colapsa todo a una sola
    partición y convierte la evaluación en el paso más caro del notebook.
    """
    cortes = pred.approxQuantile(col_p, [i / n for i in range(1, n)], 0.001)
    cortes = sorted(set(cortes))
    expr = F.lit(0)
    for c in cortes:
        expr = expr + (F.col(col_p) > F.lit(c)).cast("int")
    d = (pred.withColumn("decil", expr)
             .groupBy("decil")
             .agg(F.count(F.lit(1)).alias("impresiones"),
                  F.sum(OBJETIVO).alias("clicks"),
                  F.avg(col_p).alias("p_media"))
             .orderBy(F.col("decil").desc())
             .toPandas())
    global_ctr = d.clicks.sum() / d.impresiones.sum()
    d["ctr_pct"] = (d.clicks / d.impresiones * 100).round(3)
    d["p_media_pct"] = (d.p_media * 100).round(3)
    d["lift"] = (d.clicks / d.impresiones / global_ctr).round(3)
    d["decil"] = d.decil.map(lambda i: f"D{n - i}")   # D1 = el 10% de mayor score
    return d[["decil", "impresiones", "clicks", "ctr_pct", "p_media_pct", "lift"]]


def curva_pr(pred, col_p: str = "p_calibrada", puntos: int = 60) -> pd.DataFrame:
    """Precisión y recall en `puntos` umbrales. Se calcula agregando por bin de
    score en una sola pasada: recorrer la tabla una vez por umbral serían 60
    recorridos completos sobre millones de filas."""
    bins = (pred.withColumn("bin", F.least(
                F.floor(F.col(col_p) * puntos), F.lit(puntos - 1)).cast("int"))
                .groupBy("bin")
                .agg(F.count(F.lit(1)).alias("n"), F.sum(OBJETIVO).alias("pos"))
                .orderBy(F.col("bin").desc())
                .toPandas())
    bins["n_acum"] = bins.n.cumsum()
    bins["pos_acum"] = bins.pos.cumsum()
    total_pos = bins.pos.sum()
    bins["precision"] = bins.pos_acum / bins.n_acum
    bins["recall"] = bins.pos_acum / total_pos if total_pos else 0.0
    bins["umbral"] = bins.bin / puntos
    return bins[["umbral", "n_acum", "precision", "recall"]]


def comparar_submuestreo(pred_completo, pred_muestra, col_p: str = "p_calibrada") -> pd.DataFrame:
    """La evidencia de que evaluar sobre la muestra engaña: AUC-ROC coincide
    (es orden puro) y AUC-PR no (depende de la prevalencia)."""
    a, b = metricas_binarias(pred_completo, col_p), metricas_binarias(pred_muestra, col_p)
    filas = []
    for k in ("ctr_observado", "auc_roc", "auc_pr", "logloss"):
        filas.append({"metrica": k, "conjunto_completo": a[k], "submuestra": b[k],
                      "diferencia_rel": round((b[k] - a[k]) / a[k], 4) if a[k] else None})
    return pd.DataFrame(filas)


# ---------------------------------------------------------------------------
# Ranking (ALS)
# ---------------------------------------------------------------------------
def metricas_ranking(recomendadas, reales, k: int = None) -> dict:
    """Precision@k, Recall@k, MAP@k y NDCG@k.

    `recomendadas` y `reales`: DataFrames con (userid, lista). Se evalúa solo
    sobre usuarios presentes en ambos; los usuarios frío —sin historia en train—
    se cuentan aparte y se reportan, porque esconderlos infla todas las métricas.
    """
    k = k or config.TOP_K
    j = recomendadas.join(reales, on="userid", how="inner").collect()
    n_rec = recomendadas.count()
    n_real = reales.count()

    p, r, ap, ndcg = [], [], [], []
    for fila in j:
        rec = list(fila["recomendadas"])[:k]
        real = set(fila["reales"])
        if not real:
            continue
        aciertos = [1 if i in real else 0 for i in rec]
        n_ac = sum(aciertos)
        p.append(n_ac / k)
        r.append(n_ac / len(real))
        acum, prec = 0, 0.0
        for i, a in enumerate(aciertos, start=1):
            if a:
                acum += 1
                prec += acum / i
        ap.append(prec / min(len(real), k))
        dcg = sum(a / math.log2(i + 1) for i, a in enumerate(aciertos, start=1))
        idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(real), k) + 1))
        ndcg.append(dcg / idcg if idcg else 0.0)

    n = len(p)
    return {
        "k": k,
        "usuarios_evaluados": n,
        "usuarios_con_verdad": n_real,
        "usuarios_con_recomendacion": n_rec,
        "cobertura_pct": round(100 * n / n_real, 2) if n_real else 0.0,
        "precision_at_k": round(sum(p) / n, 6) if n else 0.0,
        "recall_at_k": round(sum(r) / n, 6) if n else 0.0,
        "map_at_k": round(sum(ap) / n, 6) if n else 0.0,
        "ndcg_at_k": round(sum(ndcg) / n, 6) if n else 0.0,
    }


# ---------------------------------------------------------------------------
# Gráficos (artefactos de MLflow)
# ---------------------------------------------------------------------------
def graficar_pr(curvas: dict, prevalencia: float, ruta: str):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 4.2), dpi=130)
    for nombre, d in curvas.items():
        ax.plot(d["recall"], d["precision"], label=nombre, lw=1.6)
    ax.axhline(prevalencia, ls="--", lw=1, color="#888",
               label=f"azar (prevalencia {prevalencia*100:.2f}%)")
    ax.set_xlabel("recall"); ax.set_ylabel("precisión")
    ax.set_title("Curva precisión-recall · día de test completo")
    ax.legend(fontsize=7); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(ruta); plt.close(fig)
    return ruta


def graficar_calibracion(deciles: pd.DataFrame, ruta: str):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5.2, 4.2), dpi=130)
    ax.plot(deciles.p_media_pct, deciles.ctr_pct, "o-", lw=1.5)
    lim = max(deciles.p_media_pct.max(), deciles.ctr_pct.max()) * 1.08
    ax.plot([0, lim], [0, lim], ls="--", lw=1, color="#888", label="calibración perfecta")
    ax.set_xlabel("CTR predicho medio (%)"); ax.set_ylabel("CTR observado (%)")
    ax.set_title("Calibración por decil de score")
    ax.legend(fontsize=8); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(ruta); plt.close(fig)
    return ruta


def graficar_curva_aprendizaje(d: pd.DataFrame, ruta: str):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax1 = plt.subplots(figsize=(6, 4.2), dpi=130)
    ax1.plot(d.filas, d.auc_pr, "o-", color="#1f77b4", label="AUC-PR")
    ax1.set_xlabel("filas de entrenamiento"); ax1.set_ylabel("AUC-PR", color="#1f77b4")
    ax2 = ax1.twinx()
    ax2.plot(d.filas, d.t_ajuste_s, "s--", color="#d62728", label="tiempo de ajuste (s)")
    ax2.set_ylabel("segundos", color="#d62728")
    ax1.set_title("¿Más datos, o más modelo? Curva de aprendizaje y su costo")
    ax1.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(ruta); plt.close(fig)
    return ruta

In [ ]:
%%writefile adbd/seguimiento.py
"""
MLflow. Backend de archivos local (`file:./mlruns`), que es lo que funciona en
Colab gratuito sin servidor ni base de datos, y se entrega comprimido junto con
el repositorio para que el tracking sea auditable y no una captura de pantalla.

Qué se registra en cada run, y por qué cada cosa:
  · parámetros  — los del modelo Y los del pipeline de datos (tasa de submuestreo,
                  m de suavizado, fechas del split). Sin estos últimos, dos runs
                  con el mismo AUC son indistinguibles y no se sabe cuál reproducir.
  · métricas    — sobre validación y sobre test, con el sufijo del conjunto.
  · artefactos  — curvas PR, calibración, tabla de deciles, importancias,
                  el registro de variables y el modelo del ganador.
  · etiquetas   — la pregunta que el experimento responde, para que la tabla
                  comparativa se lea sin tener que abrir el código.
"""
from __future__ import annotations

import json
import os

import mlflow

from . import config


def iniciar(nombre_experimento: str | None = None, ruta: str | None = None) -> str:
    ruta = ruta or config.RUTA_MLRUNS
    os.makedirs(ruta, exist_ok=True)
    mlflow.set_tracking_uri(f"file:{os.path.abspath(ruta)}")
    exp = nombre_experimento or config.NOMBRE_EXPERIMENTO
    mlflow.set_experiment(exp)
    print(f"MLflow · tracking en {mlflow.get_tracking_uri()} · experimento '{exp}'")
    return exp


def parametros_datos(info_muestra: dict, extra: dict | None = None) -> dict:
    """Los parámetros del PIPELINE DE DATOS, que son los que de verdad hacen
    reproducible un run. El modelo es la parte fácil de repetir."""
    p = {
        "split_burnin": config.DIA_BURNIN,
        "split_train": f"{config.TRAIN_INI}..{config.TRAIN_FIN}",
        "split_val": config.DIA_VAL,
        "split_test": config.DIA_TEST,
        "m_suavizado": config.M_SUAVIZADO,
        "m_suavizado_usuario": config.M_SUAVIZADO_USUARIO,
        "entidades_historicas": ",".join(config.ENTIDADES_HISTORICAS),
        "tasa_negativos_nominal": info_muestra.get("tasa_nominal"),
        "tasa_negativos_efectiva": round(info_muestra.get("tasa_efectiva", 1.0), 6),
        "filas_entrenamiento": info_muestra.get("filas_entrenamiento"),
        "prevalencia_muestra": round(info_muestra.get("prevalencia_muestra", 0), 6),
        "prevalencia_real": round(info_muestra.get("prevalencia_real", 0), 6),
        "semilla": config.SEMILLA,
    }
    p.update(extra or {})
    return p


def registrar_run(nombre, pregunta, parametros, metricas, artefactos=None,
                  tablas=None, modelo=None, ruta_artefactos=None):
    """Un run completo. Devuelve el run_id para poder citarlo en el informe."""
    ruta_artefactos = ruta_artefactos or config.RUTA_ARTEFACTOS
    os.makedirs(ruta_artefactos, exist_ok=True)
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.set_tag("pregunta", pregunta)
        mlflow.set_tag("fase", "2")
        mlflow.log_params({k: v for k, v in parametros.items() if v is not None})
        mlflow.log_metrics({k: float(v) for k, v in metricas.items()
                            if v is not None and isinstance(v, (int, float))})
        for a in (artefactos or []):
            if a and os.path.exists(a):
                mlflow.log_artifact(a)
        for nombre_tabla, df in (tablas or {}).items():
            ruta = os.path.join(ruta_artefactos, f"{nombre}__{nombre_tabla}.csv")
            df.to_csv(ruta, index=False)
            mlflow.log_artifact(ruta)
        if modelo is not None:
            try:
                from mlflow import spark as mlflow_spark

                mlflow_spark.log_model(modelo, artifact_path="modelo")
            except Exception as e:   # en Colab gratuito el guardado puede quedarse sin disco
                mlflow.set_tag("modelo_no_registrado", f"{type(e).__name__}: {str(e)[:150]}")
        return run.info.run_id


def tabla_comparativa(experimento: str | None = None):
    """La comparación de experimentos que pide la pauta, leída DESDE MLflow y no
    desde variables en memoria: si la tabla del informe sale del tracking, el
    tracking es la fuente de verdad y no una decoración."""
    import pandas as pd

    exp = mlflow.get_experiment_by_name(experimento or config.NOMBRE_EXPERIMENTO)
    if exp is None:
        return pd.DataFrame()
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    if runs.empty:
        return runs
    cols = ["tags.mlflow.runName", "tags.pregunta",
            "metrics.test_auc_pr", "metrics.test_auc_roc", "metrics.test_logloss",
            "metrics.test_lift_d1", "metrics.val_auc_pr",
            "metrics.t_ajuste_final_s", "metrics.t_busqueda_s", "run_id"]
    presentes = [c for c in cols if c in runs.columns]
    d = runs[presentes].copy()
    d.columns = [c.replace("tags.mlflow.runName", "experimento")
                  .replace("tags.", "").replace("metrics.", "") for c in presentes]
    if "test_auc_pr" in d.columns:
        d = d.sort_values("test_auc_pr", ascending=False)
    return d.reset_index(drop=True)


def guardar_json(obj, ruta: str) -> str:
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    return ruta

In [ ]:
%%writefile adbd/__init__.py
"""
adbd · Proyecto Integrador de Análisis de Big Data · Magíster en Data Science UDD
Equipo: Juan José Torres · Claudio Ballerini · Cristian Vargas · Christian Vásquez

Fase 2 — pipeline batch + ML distribuido + MLflow.
Las capas siguen la arquitectura decidida y medida en la Fase 1:

    CSV crudo -> BRONZE (Parquet+ZSTD por fecha_local) -> SILVER -> GOLD -> modelos

Cada módulo tiene una sola responsabilidad y se puede correr aislado, que es lo
que hace posible el `scripts/correr_fase2.py` de punta a punta.
"""

__version__ = "2.0.0"

from . import (  # noqa: F401
    config,
    contrato,
    bronze,
    silver,
    gold,
    transformadores,
    features,
    modelos,
    evaluacion,
    seguimiento,
    utilidades,
)

In [ ]:
import importlib, sys
sys.path.insert(0, os.path.abspath("."))
import adbd
importlib.reload(adbd)
from adbd import (bronze, config, evaluacion, features, gold as gold_mod,
                  modelos, seguimiento, silver as silver_mod, utilidades)
from adbd.transformadores import CorrectorPrior
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

config.RUTA_CSV = RUTA_CSV
RES, crono = {}, utilidades.Crono()
print("paquete adbd", adbd.__version__, "listo")

In [ ]:
spark = utilidades.crear_sesion("ADBD-Fase2")
RES["entorno"] = utilidades.huella_entorno(spark)
RES["entorno"]

## 1. Bronze · la capa que ya existe

La Fase 1 dejó 26.557.961 impresiones en Parquet+ZSTD particionado por `fecha_local`, 376,2 MB en
8 particiones; la de esta fase agrega `ts_local`, `hora_local` y `franja` y por eso pesa **435,0 MB**.
**No se reescribe**: se verifica que cumpla el contrato y se reutiliza. Que el
contrato se **verifique** en vez de suponerse no es ceremonia — un Parquet de una corrida anterior
con el esquema incompleto sobrevive en disco y el pipeline correría igual, fallando recién en la
consulta que usa la columna que falta.

`rehacer=True` reconstruye todo desde los CSV: es la reproducibilidad de extremo a extremo que la
Fase 3 va a exigir, disponible desde ahora.

In [ ]:
with crono.medir("bronze"):
    RES["bronze"] = bronze.construir(spark, rehacer=False)

imp, ads, usr = bronze.leer(spark)
RES["filas_bronze"] = imp.count()
RES["n_avisos"], RES["n_perfiles"] = ads.count(), usr.count()
print(f"impresiones {RES['filas_bronze']:,} · avisos {RES['n_avisos']:,} · perfiles {RES['n_perfiles']:,}")

## 2. Silver · calidad, y las dos decisiones que la Fase 1 dejó pendientes

La Fase 1 midió `pvalue_level` **54,24%** de nulos y `new_user_class_level` **32,49%**, y escribió
que se decidían acá. La respuesta refleja sería imputar con la moda. Antes de decidir se mide algo
concreto: **¿el grupo sin dato clickea distinto del grupo con dato?**

Si la respuesta es sí, "no sé" es información sobre el usuario y sustituirla por la moda la
destruye. Es la misma lógica que en la Fase 1 salvó al segmento sin perfil: filtrarlo era la
decisión obvia y habría borrado el segundo grupo más grande, que además clickea **por encima** del
promedio (5,33% contra 5,14%).

Y la completitud que importa es **la del cruce**, no la de la tabla de perfiles: dentro de
`user_profile.csv` varias columnas tienen 0% de nulos, pero en el JOIN falta el 5,76%. Por la misma
razón los porcentajes de la Fase 1 no se repiten aquí: `pvalue_level` tenía 54,24% de nulos *dentro
de la tabla* y tiene **54,80%** en el cruce; `new_user_class_level`, 32,49% contra **30,97%**.

**Qué decide "informativa", y por qué no es el tamaño del efecto.** La primera versión de esta
regla usaba un umbral fijo sobre la razón de CTR (5%). Sobre 26,6M de filas ese umbral está mal
calibrado en las dos direcciones: el grupo sin perfil tiene 5,335% de CTR contra 5,132% del grupo
con perfil —razón 1,040, *por debajo* del umbral— y sin embargo la diferencia tiene **z = 11,0**.
Con el umbral viejo las ocho columnas salían marcadas *"ausencia no informativa"*, contradiciendo
el hallazgo de la Fase 1. Lo que la regla necesita saber es si la diferencia **existe**, y eso es
una prueba de dos proporciones; el tamaño del efecto se reporta aparte para que el lector juzgue si
además importa.

In [ ]:
with crono.medir("perfilamiento de calidad"):
    perfil = silver_mod.perfilar_calidad(spark)
    decisiones = silver_mod.decidir_imputacion(perfil)
RES["calidad"] = decisiones.to_dict("records")
decisiones

**Control heredado: la cardinalidad antes de leer cualquier tasa.** Un JOIN mal especificado sobre
`adgroup_id` infla las filas y el CTR resultante *sigue pareciendo razonable*. El `assert` detiene
el pipeline si algún join mueve el universo.

In [ ]:
with crono.medir("silver"):
    card = silver_mod.validar_cardinalidad(spark, RES["filas_bronze"])
    silver = silver_mod.construir(spark, decisiones)
    RES["filas_silver"] = silver.count()
RES["cardinalidad"] = card.to_dict("records")
print(f"Silver: {RES['filas_silver']:,} filas")
card

## 3. Gold · *features*, y el riesgo que define esta fase

Aquí se gana o se pierde la Fase 2, porque aquí vive la **fuga de información**. La Fase 1 ya
encontró una: `CURRENT ROW` en lugar de `1 PRECEDING` en W3 metía el resultado a predecir dentro
del *feature*, y el split temporal seguía viéndose impecable. Un AUC inflado no falla, miente.

**Regla única:** ningún *feature* puede depender de una fila cuyo `clk` el modelo todavía no habría
visto en el instante de la impresión.

De ahí salen las tres familias de *features* derivados del objetivo:

| Familia | Cómo se difiere | Qué captura |
|---|---|---|
| CTR histórico por entidad (`cate_id`, `campaign_id`, `customer`, `pid`) | ventana expansiva con **retardo de un día**: para el día *d* solo suma días *< d* | calidad del inventario |
| CTR y fatiga del usuario | ventana expansiva **dentro del flujo**, `rowsBetween(unboundedPreceding, -1)`, con el desempate total de W2 | el hallazgo más accionable de la Fase 1: el CTR cae de 7,01% en la primera impresión a 4,30% de la vigésima |
| CTR móvil del slot | `RANGE BETWEEN 6 PRECEDING AND 1 PRECEDING`, literal de W3 | tendencia reciente de la posición |

Todos suavizados por m-estimación, `(clicks_previos + m·p₀) / (impresiones_previas + m)`, con el
prior `p₀` **también diferido**. Una entidad nueva recibe el prior, no un `NaN` ni un cero engañoso.

**El costo de la regla es un día.** El `06-may` no se entrena: existe para que el `07-may` tenga
historia. Se paga y se declara.

In [ ]:
with crono.medir("gold"):
    gold = gold_mod.construir(spark, silver)
    RES["filas_gold"] = gold.count()

resumen = gold_mod.resumen_split(gold)
RES["split"] = resumen.to_dict("records")
print("El CTR por día tiene que ser estable: si el día de test tuviera un CTR muy distinto,")
print("comparar métricas entre validación y test no significaría nada.\n")
resumen

### 3.1 El control de fuga, ejecutable

Tres pruebas. Si alguna falla el notebook **se detiene**: publicar métricas de un modelo con fuga
es peor que no tener modelo.

1. Ninguna correlación `|r| > 0,95` entre un *feature* numérico y el objetivo.
2. El primer día entrenable no tiene históricos nulos — el burn-in cumplió su función.
3. Los CTR históricos caen dentro de `(0, 1]`.

In [ ]:
with crono.medir("control de fuga"):
    fuga = gold_mod.verificar_fuga(gold, spark)
RES["fuga"] = fuga.to_dict("records")
assert (fuga.veredicto == "OK").all(), "control de fuga NO superado; no se sigue"
fuga

### 3.2 Registro de variables versionado (`gold_ads_v1`)

El artefacto que la Fase 1 prometió: lo aceptado **y lo descartado con su motivo**. Lo segundo es lo
que evita que en dos semanas alguien reincorpore `brand` sin saber que sus nulos los fabricó la
línea de carga.

In [ ]:
registro = gold_mod.registro_variables()
RES["registro_variables"] = registro.to_dict("records")
print(f"{(registro.estado=='aceptada').sum()} variables aceptadas · "
      f"{(registro.estado=='DESCARTADA').sum()} descartadas con motivo\n")
registro[registro.estado == "DESCARTADA"][["variable", "motivo"]]

In [ ]:
train, val, test = gold_mod.particionar(gold)
for d in (train, val, test):
    d.cache()
RES["n_train"], RES["n_val"], RES["n_test"] = train.count(), val.count(), test.count()
print(f"train {RES['n_train']:,} · val {RES['n_val']:,} · test {RES['n_test']:,}")

## 4. Desbalance: submuestreo de negativos y su precio

Con CTR 5,19% en la ventana de entrenamiento (≈1:18), entrenar sobre sus **16.744.897 filas** con
`CrossValidator` de 3 pliegues y las 12 combinaciones que suman las cuatro grillas son **36 ajustes
completos**. Se conserva **1 de cada 10 negativos y todos los positivos**, y el entrenamiento baja a
**2.458.575 filas**.

El atajo tiene dos costos y los dos se pagan explícitamente:

1. **La probabilidad queda inflada.** El modelo aprende sobre una prevalencia inventada. La
   corrección es exacta: `p = p_s·r / (p_s·r + 1 − p_s)`, implementada como un Transformer
   (`CorrectorPrior`) y probada contra casos cerrados en `tests/test_humo.py`.
2. **La evaluación no puede hacerse sobre la muestra.** AUC-PR depende de la prevalencia. La
   sección 8 lo demuestra con números en vez de afirmarlo.

Se usa la tasa **efectiva** —la fracción realmente conservada— y no la nominal: `sample` es de
Bernoulli por fila y con la nominal la corrección queda sesgada en el tercer decimal.

In [ ]:
with crono.medir("submuestreo de negativos"):
    train_ds, info_ds = modelos.submuestrear_negativos(train)
    train_ds = train_ds.cache(); train_ds.count()
RES["submuestreo"] = info_ds
r_efectiva = info_ds["tasa_efectiva"]
corrector = CorrectorPrior(inputCol="probability", outputCol="p_calibrada",
                           tasaNegativos=r_efectiva)
pd.DataFrame([info_ds]).T.rename(columns={0: "valor"})

## 5. MLflow y la línea base

`file:./mlruns` — backend de archivos, que es lo que funciona en Colab gratuito sin servidor ni base
de datos. Se entrega comprimido con el repositorio para que el *tracking* sea auditable.

En cada run se registran los parámetros del **modelo** y los del **pipeline de datos** (tasa de
submuestreo, `m` de suavizado, fechas del split). Sin los segundos, dos runs con el mismo AUC son
indistinguibles y no se sabe cuál reproducir.

**La línea base primero.** Predecir siempre el CTR histórico no es relleno: fija el piso. Con
prevalencia 5,14%, un clasificador que dice siempre "no click" tiene 94,86% de *accuracy* y cero
valor, y su AUC-PR es exactamente la prevalencia. Sin este número no se puede leer ningún otro.

In [ ]:
seguimiento.iniciar()
params_datos = seguimiento.parametros_datos(info_ds)

p0, base_test = modelos.linea_base_prior(train, test)
m_base = evaluacion.metricas_binarias(base_test)
RES["linea_base"] = {"p0": p0, **m_base}
seguimiento.registrar_run(
    "baseline_prior", "¿Cuál es el piso real? Predecir siempre el CTR histórico.",
    {**params_datos, "modelo": "constante", "p0": round(p0, 6)},
    {f"test_{k}": v for k, v in m_base.items() if isinstance(v, (int, float))})
print(f"línea base · AUC-PR {m_base['auc_pr']:.5f} = prevalencia {m_base['ctr_observado']:.5f} "
      f"· AUC-ROC {m_base['auc_roc']:.3f} (0,5 por construcción)")

## 6. Los experimentos

Cuatro, y **cada uno responde una pregunta**. Un barrido de modelos sin pregunta produce una tabla
que no se puede defender ante el panel.

| Experimento | Pregunta |
|---|---|
| `lr_contexto` | ¿Cuánto se predice **sin historia**, solo con perfil y contexto? Aísla el aporte real de los *features* derivados del objetivo. |
| `lr_completo` | ¿Cuánto agregan los CTR históricos diferidos y la fatiga? |
| `rf_completo` | ¿Gana un *ensemble* de árboles sin interacciones explícitas? |
| `gbt_completo` | ¿Compensa el *boosting* su costo de cómputo en AUC-PR? |

**Sobre el `CrossValidator` y el split temporal.** El k-fold aleatorio parte la ventana de
entrenamiento sin respetar el orden, lo que normalmente sería una fuga. Aquí no lo es, porque la
protección temporal vive en los **features** —`gold.py` los construye diferidos— y no en el corte:
una fila del 10-may no contiene nada del 11-may aunque caiga en el mismo pliegue. La afirmación no
se deja en palabras: se mide la **brecha de optimismo** contra el día de validación retenido.

**Y esa brecha hay que medirla bien.** El `CrossValidator` calcula su AUC-PR sobre pliegues ya
submuestreados (prevalencia 35,4%) y la validación va sobre el día completo (5,0%). Restar esos dos
números da una diferencia enorme que **no mide optimismo sino prevalencia**. La comparación válida
es contra una validación submuestreada a la misma tasa, con el contraste en AUC-ROC —invariante al
submuestreo— como control cruzado.

**Presupuesto de cómputo, declarado.** La *búsqueda* corre sobre el 20% del train ya submuestreado;
el modelo **ganador** se reajusta sobre el train submuestreado completo.

In [ ]:
numericas_sin_historia = [c for c in gold_mod.NUMERICAS
                          if not c.startswith(("ctr_hist_", "log_imp_hist_", "ctr_usuario",
                                               "log_imp_previas", "ctr_movil"))]
catalogo = modelos.catalogo_experimentos(numericas_sin_historia)
resultados, curvas, modelos_ajustados = {}, {}, {}

for nombre, spec in catalogo.items():
    print(f"\n===== {nombre} · {spec['pregunta']}")
    with crono.medir(f"experimento {nombre}"):
        modelo, info_cv = modelos.ajustar_con_cv(train_ds, spec)

    pv = corrector.transform(modelo.transform(val))
    pt = corrector.transform(modelo.transform(test))
    m_val, m_test = evaluacion.metricas_binarias(pv), evaluacion.metricas_binarias(pt)
    deciles = evaluacion.tabla_deciles(pt)
    curvas[nombre] = evaluacion.curva_pr(pt)
    lift_d1 = float(deciles.iloc[0]["lift"])

    val_ds, _ = modelos.submuestrear_negativos(val, r_efectiva)
    m_val_ds = evaluacion.metricas_binarias(corrector.transform(modelo.transform(val_ds)))
    brecha = info_cv["cv_mejor"] - m_val_ds["auc_pr"]
    brecha_roc = m_val_ds["auc_roc"] - m_val["auc_roc"]

    metricas = ({f"val_{k}": v for k, v in m_val.items() if isinstance(v, (int, float))}
                | {f"test_{k}": v for k, v in m_test.items() if isinstance(v, (int, float))}
                | {"test_lift_d1": lift_d1, "cv_auc_pr": info_cv["cv_mejor"],
                   "val_ds_auc_pr": m_val_ds["auc_pr"],
                   "brecha_optimismo_cv_val": brecha,
                   "brecha_auc_roc_muestra_vs_completo": brecha_roc,
                   "t_busqueda_s": info_cv["t_busqueda_s"],
                   "t_ajuste_final_s": info_cv["t_ajuste_final_s"]})
    run_id = seguimiento.registrar_run(
        nombre, spec["pregunta"],
        {**params_datos, **info_cv["mejores_parametros"],
         "estimador": type(spec["estimador"]).__name__,
         "escalado": spec.get("escalar", False),
         "n_combinaciones": info_cv["n_combinaciones"], "folds": info_cv["folds"],
         "fraccion_busqueda": info_cv["fraccion_busqueda"]},
        metricas, tablas={"deciles": deciles, "curva_pr": curvas[nombre]})

    imp = features.importancias(modelo, numericas=spec.get("numericas"))
    modelos_ajustados[nombre] = modelo
    resultados[nombre] = {"run_id": run_id, "cv": info_cv, "val": m_val, "test": m_test,
                          "val_submuestreada": m_val_ds, "lift_d1": lift_d1,
                          "brecha_optimismo": brecha, "brecha_auc_roc": brecha_roc,
                          "importancias": imp.to_dict("records"),
                          "deciles": deciles.to_dict("records")}
    print(f"  AUC-PR test {m_test['auc_pr']:.5f} · AUC-ROC {m_test['auc_roc']:.5f} "
          f"· lift D1 {lift_d1:.2f}x · brecha CV-val {brecha:+.5f}")

RES["experimentos"] = resultados
RES["mejor_experimento"] = max(resultados, key=lambda k: resultados[k]["test"]["auc_pr"])
print("\nmejor por AUC-PR en test:", RES["mejor_experimento"])

### 6.1 Qué mueve la predicción

Los coeficientes y las importancias sirven para dos cosas distintas y las dos importan en la
defensa: explicar el modelo, y **detectar fuga que las métricas no delatan**. Una variable que
concentra casi toda la importancia suele ser una fuga, no un hallazgo.

Los nombres se leen de `categorySizes` del `OneHotEncoderModel` ajustado y no se deducen a mano:
deducirlos desalinea los nombres por una posición y hace que el informe atribuya un coeficiente a la
variable equivocada — un error que no se ve porque no falla.

In [ ]:
mejor = RES["mejor_experimento"]
imp = pd.DataFrame(resultados[mejor]["importancias"])
print(f"{mejor} · top 15 variables\n")
imp.head(15)

### 6.2 El contraste: ¿cuánto cuesta el atajo?

El mismo modelo sobre el train **completo**, sin submuestrear, con `weightCol`. Sin este
experimento, "submuestreamos por costo" es una afirmación sin número detrás.

Un detalle que el experimento mismo enseña: `weightCol` **tampoco** deja la probabilidad calibrada.
Pesar los positivos por *w* multiplica sus *odds* por *w*, igual que el submuestreo las multiplica
por `1/r`. Es la misma corrección con `r = 1/w`, y omitirla sería el error que este experimento
existe para desmentir.

In [ ]:
spec = catalogo["lr_completo"]
with crono.medir("experimento lr_pesos_completo"):
    train_w, info_w = modelos.agregar_pesos(train)
    est = spec["estimador"].copy(); est.setWeightCol("peso")
    pipe_w = features.pipeline(est, numericas=spec.get("numericas"), escalar=spec.get("escalar"))
    t0 = time.perf_counter(); modelo_w = pipe_w.fit(train_w); t_w = round(time.perf_counter() - t0, 1)

corrector_w = CorrectorPrior(inputCol="probability", outputCol="p_calibrada",
                             tasaNegativos=1.0 / info_w["peso_positivo"])
pt_w = corrector_w.transform(modelo_w.transform(test))
m_test_w, dec_w = evaluacion.metricas_binarias(pt_w), evaluacion.tabla_deciles(pt_w)

RES["contraste_pesos"] = {**info_w, "t_ajuste_final_s": t_w, "test": m_test_w,
                          "lift_d1": float(dec_w.iloc[0]["lift"]),
                          "delta_auc_pr_vs_submuestreo":
                              round(m_test_w["auc_pr"] - resultados["lr_completo"]["test"]["auc_pr"], 6),
                          "razon_tiempo": round(t_w / max(resultados["lr_completo"]["cv"]["t_ajuste_final_s"], 1e-9), 2)}
seguimiento.registrar_run(
    "lr_pesos_completo",
    "¿Cuánto AUC-PR cuesta el submuestreo frente a entrenar con el dataset completo?",
    {**params_datos, "estimador": "LogisticRegression", "weightCol": "peso",
     "submuestreo": "no", "peso_positivo": info_w["peso_positivo"]},
    {f"test_{k}": v for k, v in m_test_w.items() if isinstance(v, (int, float))}
    | {"t_ajuste_final_s": t_w, "test_lift_d1": float(dec_w.iloc[0]["lift"])},
    tablas={"deciles": dec_w})

pd.DataFrame([
    {"variante": "submuestreo 1:10 + recalibración",
     "filas": info_ds["filas_entrenamiento"],
     "t_ajuste_s": resultados["lr_completo"]["cv"]["t_ajuste_final_s"],
     "auc_pr_test": resultados["lr_completo"]["test"]["auc_pr"],
     "logloss": resultados["lr_completo"]["test"]["logloss"]},
    {"variante": "train completo + weightCol",
     "filas": info_w["filas"], "t_ajuste_s": t_w,
     "auc_pr_test": m_test_w["auc_pr"], "logloss": m_test_w["logloss"]},
])

## 7. Escalamiento: ¿más datos o más modelo?

La Fase 1 dejó una pregunta abierta con nombre propio: **¿conviene incorporar `behavior_log`**
(~704M de registros, 26× el volumen) que el espejo de Kaggle no trae? La respuesta no es una
intuición: se ajusta el mejor modelo sobre fracciones crecientes del entrenamiento y se mira si la
curva de aprendizaje **ya está plana**.

Si lo está, 26× más volumen compra tiempo de cómputo y no precisión, y eso es una decisión de
escalamiento defendible con un número. El tiempo de ajuste en el eje derecho es la otra mitad del
argumento: dice si el costo crece lineal con las filas o peor.

**Cuidado con lo que la curva NO dice.** Si sale plana, lo está *para este espacio de features y este
modelo*. `behavior_log` no traería solo más filas de lo mismo: traería el comportamiento de
navegación, que son *features* que hoy no existen. La conclusión honesta es «más filas de las mismas
variables no compran precisión», no «más datos nunca sirven».

In [ ]:
with crono.medir("curva de aprendizaje"):
    d_curva = modelos.curva_aprendizaje(
        train_ds, catalogo[RES["mejor_experimento"]],
        fracciones=(0.1, 0.25, 0.5, 1.0), evaluar=val, corrector=corrector,
        metrica_fn=evaluacion.metricas_binarias)
RES["curva_aprendizaje"] = d_curva.to_dict("records")
evaluacion.graficar_curva_aprendizaje(d_curva, "artefactos_fase2/curva_aprendizaje.png")
d_curva

## 8. Por qué la evaluación va sobre el conjunto completo

La demostración, con números. El mismo modelo, evaluado sobre el día de test completo y sobre el
mismo día submuestreado a la tasa de entrenamiento:

- **AUC-ROC apenas se mueve**: es puro orden, y el submuestreo de negativos no reordena nada.
- **AUC-PR se dispara**: depende de la prevalencia por definición.

Reportar el segundo número sería reportar un modelo que no existe.

In [ ]:
m = modelos_ajustados[RES["mejor_experimento"]]
test_ds, _ = modelos.submuestrear_negativos(test, r_efectiva)
comp = evaluacion.comparar_submuestreo(corrector.transform(m.transform(test)),
                                       corrector.transform(m.transform(test_ds)))
RES["efecto_submuestreo_en_la_evaluacion"] = comp.to_dict("records")
comp

### 8.1 La traducción a negocio: *lift* por decil

El panel de la defensa no compra un AUC. En una plataforma de display no se decide "click o no
click": se **ordena inventario**. Lo que importa es cuánto mejor rinde el 10% de impresiones que el
modelo pone arriba (D1) comparado con servir al azar, y si la probabilidad está lo bastante
calibrada como para poner un umbral de negocio sobre ella.

In [ ]:
deciles = pd.DataFrame(resultados[RES["mejor_experimento"]]["deciles"])
evaluacion.graficar_pr(curvas, RES["linea_base"]["ctr_observado"], "artefactos_fase2/curva_pr.png")
evaluacion.graficar_calibracion(deciles, "artefactos_fase2/calibracion.png")
print(f"D1 rinde {deciles.iloc[0]['lift']:.2f}x el CTR global · "
      f"D10 rinde {deciles.iloc[-1]['lift']:.2f}x\n")
deciles

## 9. ALS · recomendación con *feedback* implícito

El segundo eje de modelamiento distribuido que la Fase 1 anticipó: matriz **usuario × categoría**
con los clicks como confianza. Solo entran clicks, porque una impresión sin click no es una señal
negativa sino ausencia de evidencia — que es exactamente el supuesto que `implicitPrefs=True`
modela.

**Contra la popularidad, o no significa nada.** En *feedback* implícito la línea base de
popularidad es sorprendentemente difícil de batir, y un MAP@10 sin ese contraste es un número
suelto. Los usuarios *fríos* —sin historia en train— se cuentan y se reportan por separado:
esconderlos infla todas las métricas, porque ALS no puede recomendarles nada personalizado y la
cobertura es parte del resultado, no una nota al pie.

In [ ]:
with crono.medir("ALS implícito"):
    m_train = modelos.matriz_implicita(train).cache()
    m_test_mat = modelos.matriz_implicita(test).cache()
    RES["als_ids"] = modelos.verificar_ids_als(m_train, m_test_mat)
    als_modelo, t_als = modelos.entrenar_als(m_train)

    recs = (als_modelo.recommendForAllUsers(config.TOP_K)
            .select(F.col("userid").cast("long").alias("userid"),
                    F.col("recommendations.cate_id").alias("recomendadas")))
    reales = (m_test_mat.groupBy("userid")
              .agg(F.collect_set(F.col("cate_id").cast("int")).alias("reales")))
    m_als = evaluacion.metricas_ranking(recs, reales)

    top = modelos.recomendaciones_populares(m_train)
    pop = reales.select("userid").withColumn("recomendadas",
                                             F.array(*[F.lit(int(x)) for x in top]))
    m_pop = evaluacion.metricas_ranking(pop, reales)

RES["als"] = {"als": m_als, "popularidad": m_pop, "t_ajuste_s": t_als,
              "rank": config.ALS_RANK, "alpha": config.ALS_ALPHA}
seguimiento.registrar_run(
    "als_implicito",
    "¿Un modelo de recomendación bate a la popularidad en exposición por categoría?",
    {**params_datos, "modelo": "ALS", "rank": config.ALS_RANK, "regParam": config.ALS_REG,
     "alpha": config.ALS_ALPHA, "implicitPrefs": True},
    {f"test_{k}": v for k, v in m_als.items() if isinstance(v, (int, float))}
    | {f"pop_{k}": v for k, v in m_pop.items() if isinstance(v, (int, float))}
    | {"t_ajuste_final_s": t_als})

pd.DataFrame([{"modelo": "ALS implícito", **m_als}, {"modelo": "popularidad", **m_pop}])

## 10. Comparación de experimentos

La tabla sale **de MLflow**, no de variables en memoria. Si la tabla del informe se lee desde el
*tracking*, el *tracking* es la fuente de verdad y no una decoración que se llenó por cumplir.

In [ ]:
comparativa = seguimiento.tabla_comparativa()
RES["comparativa_mlflow"] = comparativa.to_dict("records")
comparativa

## 11. Tiempo y costo (FinOps)

Misma tarifa que la Fase 1 —**US$ 0,27/hora**, e2-standard-8 *on-demand*— para que las dos fases se
valoricen en la misma unidad. El costo monetario efectivo sigue siendo **US$ 0**.

**Pero los tiempos de las dos fases NO son comparables entre sí, y hay que decirlo antes de la
tabla.** El *benchmark* de la Fase 1 corrió en 2 núcleos con 13,6 GB de RAM; esta fase corrió en la
máquina que imprime la sección 0 (22 núcleos, 102,6 GB). Comparar 51,6 min de aquí contra 26,0 min
de allá no mide progreso ni regresión: mide dos máquinas distintas. Lo que sí es comparable es el
**reparto interno** de cada fase —dónde se va el tiempo— y el costo por hora, que es el mismo.

In [ ]:
tiempos = crono.tabla()
RES["tiempos"] = crono.marcas
RES["t_total_s"] = crono.total()
RES["usd_referencial"] = utilidades.usd(crono.total())
RES["extrapolacion_behavior_log"] = {
    "factor": round(config.FACTOR_BEHAVIOR, 1),
    "segundos_lineales": round(crono.total() * config.FACTOR_BEHAVIOR, 1),
    "usd_lineales": utilidades.usd(crono.total() * config.FACTOR_BEHAVIOR),
    "nota": ("la extrapolación es LINEAL y por eso es un piso: 26x el volumen no cabe en RAM "
             "y un motor de un solo nodo se degrada de forma no lineal"),
}
print(f"TOTAL {RES['t_total_s']:,} s ({RES['t_total_s']/60:.1f} min) · "
      f"US$ {RES['usd_referencial']} de cómputo equivalente\n")
tiempos

## 12. Exportación

Cada cifra del informe, con su clave. `resultados_fase2.json` es lo que hace verificable la regla
del equipo: *ninguna afirmación se escribe sin una celda que la imprima*.

In [ ]:
RES["notebook"] = "fase2_pipeline_ml_mlflow.ipynb"
utilidades.exportar(RES, "resultados_fase2.json")
shutil.make_archive("mlruns_fase2", "zip", config.RUTA_MLRUNS)
print("mlruns_fase2.zip escrito (tracking completo para auditar)\n")

for k in ("filas_bronze", "filas_silver", "filas_gold", "n_train", "n_val", "n_test",
          "mejor_experimento", "t_total_s", "usd_referencial"):
    print(f"  {k:<22} = {RES.get(k)}")
print()
for n, r in RES["experimentos"].items():
    print(f"  {n:<14} AUC-PR {r['test']['auc_pr']:.5f} · AUC-ROC {r['test']['auc_roc']:.5f} "
          f"· lift D1 {r['lift_d1']:.2f}x")

## 13. Declaración de uso de IA generativa

**Sí se usaron asistentes de IA.** Herramienta: **Claude (Anthropic), en Claude Code**.

| Dónde | Uso | Cómo se validó |
|---|---|---|
| Diseño del control de fuga | Primera versión de las ventanas diferidas y del suavizado por m-estimación | El equipo agregó `verificar_fuga()` como `assert` que detiene el pipeline, y el día de burn-in, que el borrador no contemplaba |
| Codificación de *features* | Ensamblado del `Pipeline` de MLlib | Se corrigió la lectura de nombres: el borrador deducía el ancho de cada bloque one-hot a mano y desalineaba los coeficientes por una posición |
| Recalibración tras submuestreo | Fórmula y su implementación como Transformer | Probada contra casos cerrados en `tests/test_humo.py`, no contra sí misma; se detectó que `weightCol` necesita la **misma** corrección y el borrador no la aplicaba |
| Catálogo de experimentos | Grillas y estructura de los runs de MLflow | El equipo exigió que cada experimento tuviera una pregunta declarada; los que no la tenían se eliminaron |
| Regla de calidad de datos | Umbrales y decisión de imputación | Reemplazada por una prueba de dos proporciones tras detectar que el umbral fijo contradecía el hallazgo de la Fase 1 sobre el segmento sin perfil |
| Redacción | Estructura y primer borrador del informe | Cada cifra sale de una celda y se exporta a `resultados_fase2.json` |

**Cuatro correcciones del equipo sobre lo que produjo el asistente.** Ninguna la habría detectado
una prueba de "¿corre?":

1. **La "brecha de optimismo" comparaba peras con manzanas.** El borrador restaba el AUC-PR del
   `CrossValidator` —calculado sobre pliegues submuestreados al ~31% de positivos— del AUC-PR de la
   validación completa al ~5%. La diferencia resultante era enorme y no medía optimismo sino
   **prevalencia**. Se corrigió midiendo contra una validación submuestreada a la misma tasa, con el
   contraste en AUC-ROC como control cruzado.
2. **`weightCol` también descalibra, y el borrador no lo corregía.** Quedaba con un *LogLoss* 3,6×
   peor que la variante submuestreada, y la conclusión "el submuestreo calibra mejor" habría sido
   falsa: el problema era la corrección faltante, no el método.
3. **Faltaba el día de burn-in.** Sin él, los *features* históricos del primer día de entrenamiento
   son nulos y la alternativa del borrador —rellenarlos con el prior global— usa el CTR de todo el
   período, incluido el futuro. Fuga silenciosa.
4. **La línea base no estaba.** El primer borrador comparaba modelos entre sí. Sin el piso
   —AUC-PR = prevalencia— ninguna de esas cifras se puede leer, y menos defender.
5. **La regla de imputación usaba un umbral fijo donde hacía falta una prueba.** "Informativa si el
   CTR difiere más de un 5%" declaraba *no informativas* las ocho columnas de perfil, cuando la
   diferencia del grupo sin perfil tiene **z = 11,0** sobre 1,5M de observaciones. La detectó el
   equipo al ver que el veredicto contradecía un hallazgo ya publicado en la Fase 1 — no un error de
   ejecución, sino un resultado que no encajaba con lo que ya se sabía.

**Las tres preguntas de la clase.** *¿Responde la pregunta real?* Sí: la pregunta es ordenar
inventario, por eso la métrica de cabecera es AUC-PR y la de negocio es el *lift* del decil
superior, no *accuracy*. *¿Se validó contra números conocidos?* Sí: el CTR reconstruido, la línea
base igual a la prevalencia por construcción, la cardinalidad de los joins con `delta = 0` y la
recalibración probada contra casos cerrados. *¿Se revisó el plan de ejecución?* La lectura de planes
fue el núcleo de la Fase 1; en la Fase 2 su equivalente es el control de fuga y la curva de
aprendizaje, que son los que dicen si el número es defendible.

---

**Anexo · Reproducibilidad.** `fase2_pipeline_ml_mlflow.ipynb` (Kernel → Restart & Run All) ·
`resultados_fase2.json` · `mlruns_fase2.zip` (tracking completo) · `artefactos_fase2/` (curvas,
deciles, importancias) · `pytest tests/ -q` verifica el pipeline completo sobre datos sintéticos en
~75 s sin descargar 1,1 GB. Lo único que varía entre corridas son los tiempos.